    ---
### 1. Préparation données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import matplotlib.dates as mdates
from matplotlib.colors import to_hex
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier as rf
#from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, auc
from sklearn.svm import SVR
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
import scipy.interpolate as spi
from scipy.interpolate import griddata
import h3
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
from collections import Counter
from scipy.stats import randint
from sklearn.dummy import DummyClassifier
import optuna
import joblib

data_firepoint = pd.read_csv('C:/Users/wittl/OneDrive/Documents/Cours/Université et Etudes/IUT Belfort/S6/Stage/data/data_full_dep/firepoint.csv') # firepoint.csv de tous les départements français

In [ ]:
import pickle
import pandas as pd

with open("C:/Users/wittl/OneDrive/Documents/Cours/Université et Etudes/IUT Belfort/S6/Stage/data/data_full_dep/df_full_departement_None_node.pkl", "rb") as file:
    data_full_dep = pickle.load(file)
# df_full_departement_None_node.pkl de tous les départements français
print(type(data_full_dep))

print(data_full_dep.head())

print(data_full_dep.columns)

print(data_full_dep.shape)

In [ ]:
data_firepoint.drop_duplicates()
data_firepoint.dropna()

"""data_full_dep.drop_duplicates()
data_full_dep.dropna()"""

In [ ]:
"""data_firepoint.select_dtypes(include=['float64', 'int64'])
data_full_dep.select_dtypes(include=['float64', 'int64'])"""

In [ ]:
import os

output_path = "../results_img_csv"
os.makedirs(output_path, exist_ok=True)

---
### 2. Analyse graphique


---
#### 2.1 Visualisation graphique feux (nb, localisation, date)

In [ ]:
data_firepoint['Date de première alerte'] = pd.to_datetime(data_firepoint['Date de première alerte'])
fires_per_day = data_firepoint.groupby(data_firepoint['Date de première alerte'].dt.date).size()

plt.figure(figsize=(12, 6))
plt.plot(fires_per_day.index, fires_per_day.values)
plt.xlabel('Date')
plt.ylabel('Nombre incendies')
plt.tight_layout()
plt.show()

In [ ]:
LAT_MIN, LAT_MAX = 41.0, 51.5
LON_MIN, LON_MAX = -5.5, 9.6

filtered_data = data_firepoint[
    (data_firepoint['latitude'] >= LAT_MIN) &
    (data_firepoint['latitude'] <= LAT_MAX) &
    (data_firepoint['longitude'] >= LON_MIN) &
    (data_firepoint['longitude'] <= LON_MAX)
]

if not filtered_data.empty:
    gridsize = 120
    x = filtered_data['longitude']
    y = filtered_data['latitude']

    counts, xedges, yedges = np.histogram2d(x, y, bins=gridsize)
    x_centers = (xedges[:-1] + xedges[1:]) / 2
    y_centers = (yedges[:-1] + yedges[1:]) / 2

    hex_centers = []
    for i in range(len(x_centers)):
        for j in range(len(y_centers)):
            if counts[i, j] > 0:
                hex_centers.append({
                    "lat": y_centers[j],
                    "lon": x_centers[i],
                    "count": counts[i, j]
                })

    map_center = [(LAT_MIN + LAT_MAX) / 2, (LON_MIN + LON_MAX) / 2]
    m = folium.Map(location=map_center, zoom_start=10)

    for hexagon in hex_centers:
        folium.CircleMarker(
            location=[hexagon["lat"], hexagon["lon"]],
            radius=5 + np.sqrt(hexagon["count"]),
            color='red',
            fill=True,
            fill_color='red',
            fill_opacity=0.6,
            tooltip=f"Incendies : {int(hexagon['count'])}"
        ).add_to(m)

    m.save('../results_img_csv/incendies_france.html')
    print("Carte enregistrée sous 'incendies_france.html'.")
else:
    print("Aucune donnée valide après nettoyage.")

In [ ]:
LAT_MIN, LAT_MAX = 41.0, 51.5
LON_MIN, LON_MAX = -5.5, 9.6

filtered_data = data_firepoint[
    (data_firepoint['latitude'] >= LAT_MIN) &
    (data_firepoint['latitude'] <= LAT_MAX) &
    (data_firepoint['longitude'] >= LON_MIN) &
    (data_firepoint['longitude'] <= LON_MAX)
]

if not filtered_data.empty:
    filtered_data['Nature'] = filtered_data['Nature'].fillna('Inconnue')
    filtered_data['Nature'] = filtered_data['Nature'].astype(str)

    grouped_data = filtered_data.groupby(['latitude', 'longitude'])['Nature'].apply(lambda x: list(set(x))).reset_index()
    grouped_data['feux_count'] = filtered_data.groupby(['latitude', 'longitude'])['Nature'].transform('count')

    nature_unique = filtered_data['Nature'].unique()
    cmap = plt.colormaps.get_cmap('tab10')
    nature_colors = {nature: to_hex(cmap(i / (len(nature_unique) - 1))) for i, nature in enumerate(nature_unique)}

    map_center = [(LAT_MIN + LAT_MAX) / 2, (LON_MIN + LON_MAX) / 2]
    m = folium.Map(location=map_center, zoom_start=6)

    for _, row in grouped_data.iterrows():
        lat, lon, natures, count = row['latitude'], row['longitude'], row['Nature'], row['feux_count']

        if len(natures) > 1:
            color = "black"
        else:
            color = nature_colors[natures[0]]

        tooltip_text = f"Natures : {', '.join(natures)}<br>Nombre de feux : {count}"

        folium.CircleMarker(
            location=[lat, lon],
            radius=6 + len(natures),
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.6,
            tooltip=tooltip_text
        ).add_to(m)

    m.save('../results_img_csv/incendies_par_nature_specifiee.html')
    print("Carte enregistrée sous 'incendies_par_nature_specifiee.html'.")
else:
    print("Aucune donnée valide après nettoyage.")

---
#### 2.2 Visualisation graphique nombre nature et feux

---
##### 2.2.1 Nb Natures et nb feux

In [ ]:
feux_par_departement = data_firepoint.groupby("Département").size().reset_index(name="Nombre de feux")
feux_par_departement = feux_par_departement.sort_values(by="Nombre de feux", ascending=False)

plt.figure(figsize=(12, 8))
plt.bar(feux_par_departement["Département"].astype(str), feux_par_departement["Nombre de feux"], color="orange")
plt.ylabel("Nombre de feux", fontsize=12)
plt.xlabel("Département", fontsize=12)
plt.title("Nombre de feux par département", fontsize=14)
plt.xticks(rotation=90, fontsize=7)
plt.show()

In [ ]:
feux_par_nature = data_firepoint.groupby("Nature").size().reset_index(name="Nombre de feux")
feux_par_nature = feux_par_nature.sort_values(by="Nombre de feux", ascending=False)

plt.figure(figsize=(10, 6))
plt.bar(feux_par_nature["Nature"].astype(str), feux_par_nature["Nombre de feux"], color="firebrick")
plt.ylabel("Nombre de feux", fontsize=12)
plt.xlabel("Nature", fontsize=12)
plt.title("Nombre de feux par nature", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.show()

In [ ]:
feux_par_cause = data_firepoint["Connaissance de la cause"].value_counts()

plt.figure(figsize=(10, 6))
feux_par_cause.plot(kind="bar", color="darkorange")

plt.xlabel("Cause", fontsize=12)
plt.ylabel("Nombre de feux", fontsize=12)
plt.title("Répartition des causes des incendies", fontsize=14)
plt.xticks(rotation=45)
plt.show()

---
#### 2.3 Probabilité de feu par période

In [ ]:
data_firepoint['Date de première alerte'] = pd.to_datetime(data_firepoint['Date de première alerte'])

data_firepoint['Année'] = data_firepoint['Date de première alerte'].dt.year

prob_fires_per_year = data_firepoint['Année'].value_counts(normalize=True).sort_index()

plt.figure(figsize=(10, 5))
sns.barplot(x=prob_fires_per_year.index, y=prob_fires_per_year.values, palette='flare')

plt.xlabel("Année")
plt.ylabel("Probabilité d'incendie")
plt.title("Probabilité d'incendie par année")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

In [ ]:
data_firepoint['Date de première alerte'] = pd.to_datetime(data_firepoint['Date de première alerte'])
data_firepoint['Mois'] = data_firepoint['Date de première alerte'].dt.month_name()
prob_fires_per_month = data_firepoint['Mois'].value_counts(normalize=True).sort_index()

mois_ordre = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
prob_fires_per_month = prob_fires_per_month.reindex(mois_ordre)

plt.figure(figsize=(10, 5))
sns.barplot(x=prob_fires_per_month.index, y=prob_fires_per_month.values, palette='flare')
plt.xlabel("Mois de l'année")
plt.ylabel("Probabilité d'incendie")
plt.title("Probabilité d'incendie par mois de l'année")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
data_firepoint['Date de première alerte'] = pd.to_datetime(data_firepoint['Date de première alerte'])
data_firepoint['Jour de l\'année'] = data_firepoint['Date de première alerte'].dt.dayofyear

count_fires_per_day = data_firepoint['Jour de l\'année'].value_counts().sort_index()
total_days_in_year = 365

prob_fires_per_day = count_fires_per_day / total_days_in_year

plt.figure(figsize=(12, 6))
plt.plot(prob_fires_per_day.index, prob_fires_per_day.values, color='b', linestyle='-', linewidth=2)
plt.xlabel("Jour de l'année")
plt.ylabel("Probabilité d'incendie")
plt.title("Probabilité d'incendie par jour de l'année")
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
data_firepoint['Date de première alerte'] = pd.to_datetime(data_firepoint['Date de première alerte'])
data_firepoint['Jour de la semaine'] = data_firepoint['Date de première alerte'].dt.day_name()
prob_fires_per_day = data_firepoint['Jour de la semaine'].value_counts(normalize=True).sort_index()

jours_ordre = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
prob_fires_per_day = prob_fires_per_day.reindex(jours_ordre)

plt.figure(figsize=(10, 5))
sns.barplot(x=prob_fires_per_day.index, y=prob_fires_per_day.values, palette='flare')
plt.xlabel("Jour de la semaine")
plt.ylabel("Probabilité d'incendie")
plt.title("Jour le plus probable pour un incendie")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
data_firepoint['Heure'] = data_firepoint['Date de première alerte'].dt.hour
prob_fires_per_hour = data_firepoint['Heure'].value_counts(normalize=True).sort_index()

plt.figure(figsize=(10, 5))
sns.barplot(x=prob_fires_per_hour.index, y=prob_fires_per_hour.values, palette='flare')
plt.xlabel("Heure de la journée")
plt.ylabel("Probabilité d'incendie")
plt.title("Probabilité d'incendie en fonction de l'heure")
plt.xticks(range(0, 24))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

---
### 3. Modélisation

---
#### 3.1 Préparation données

In [ ]:
cems_variables = ['temp',
                  'dwpt', 'rhum', 'prcp', 'wdir', 'wspd', 'prec24h',
                'dc', 'ffmc', 'dmc', 'nesterov', 'munger', 'kbdi',
                'isi', 'angstroem', 'bui', 'fwi', 'dailySeverityRating',
                'temp16',
                'dwpt16', 'rhum16', 'prcp16', 'wdir16', 'wspd16', 'prec24h16',
                'days_since_rain', 'sum_consecutive_rainfall', 'sum_rain_last_7_days',
                'sum_snow_last_7_days', 'snow24h', 'snow24h16'
                ]

air_variables = ['O3', 'NO2', 'PM10', 'PM25']

# Encoder
sentinel_variables = ['NDVI', 'NDMI', 'NDBI', 'NDSI', 'NDWI']
landcover_variables = [
                      'foret_encoder',
                      'argile_encoder',
                      'id_encoder',
                      'cosia_encoder'
                        ]

cluster_encoder = ['cluster_encoder']

calendar_variables = ['month', 'dayofyear', 'dayofweek', 'isweekend', 'couvrefeux', 'confinemenent',
                    'ramadan', 'bankHolidays', 'bankHolidaysEve', 'holidays', 'holidaysBorder',
                    'calendar_mean', 'calendar_min', 'calendar_max', 'calendar_sum']

geo_variables = ['departement_encoder']
region_variables = ['region_class']

# Percentage
foret_variables = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21']
cosia_variables = [
    'Other',
    'Building',
    'Bare soil',
    'Water surface',
    'Conifer',
    'Deciduous',
    'Shrubland',
    'Lawn',
    'Crop'
]

osmnx_variables = ['0', '1', '2', '3', '4', '5']
dynamic_world_variables = ['water', 'tree', 'grass', 'crops', 'shrub', 'flooded', 'built', 'bare', 'snow']

# Influence
historical_variables = ['pastinfluence']
auto_regression_variable_reg = ['J-1',
                                 #'J-2', 'J-3', 'J-4', 'J-5', 'J-6', 'J-7'
                                ]
auto_regression_variable_bin = ['B-1',
                                #'B-2', 'B-3', 'B-4', 'B-5', 'B-6', 'B-7'
                                ]

# Other
elevation_variables = ['elevation']
population_variabes = ['population']
vigicrues_variables = ['12',
                       #'16'
                      ]

nappes_variables = ['niveau_nappe_eau', 'profondeur_nappe']

# Time varying
varying_time_variables = ['temp_mean', 'dwpt_mean', 'rhum_mean', 'wdir_mean', 'wspd_mean',
                            'dc_mean', 'ffmc_mean', 'dmc_mean', 'nesterov_mean', 'munger_mean', 'kbdi_mean',
                            'isi_mean', 'angstroem_mean', 'bui_mean', 'fwi_mean', 'dailySeverityRating_mean',
                            'temp16_mean', 'dwpt16_mean', 'rhum16_mean', 'wdir16_mean', 'wspd16_mean',
                            #'air_mean',
                            #'Calendar_mean',

                            'temp_min', 'dwpt_min', 'rhum_min', 'wdir_min', 'wspd_min',
                            'dc_min', 'ffmc_min', 'dmc_min', 'nesterov_min', 'munger_min', 'kbdi_min',
                            'isi_min', 'angstroem_min', 'bui_min', 'fwi_min', 'dailySeverityRating_min',
                            'temp16_min', 'dwpt16_min', 'rhum16_min', 'wdir16_min', 'wspd16_min',
                            #'air_min',
                            #'Calendar_min',

                            'temp_max', 'dwpt_max', 'rhum_max', 'wdir_max', 'wspd_max',
                            'dc_max', 'ffmc_max', 'dmc_max', 'nesterov_max', 'munger_max', 'kbdi_max',
                            'isi_max', 'angstroem_max', 'bui_max', 'fwi_max', 'dailySeverityRating_max',
                            'temp16_max', 'dwpt16_max', 'rhum16_max', 'wdir16_max', 'wspd16_max',
                            #'air_max',
                            #'Calendar_max',

                            #'Historical_sum',
                            #'Historical_grad',
                            #'AutoRegressionBin_sum',
                            #'AutoRegressionReg_sum',
                            #'AutoRegressionReg_grad'
                            ]

varying_time_variables_name = []

train_features = [
                    'temp', 'dwpt', 'rhum', 'prcp', 'wdir', 'wspd', 'prec24h',
                    'dc', 'ffmc', 'dmc', 'nesterov', 'munger', 'kbdi',
                    'isi', 'angstroem', 'bui', 'fwi', 'dailySeverityRating',
                    'temp16', 'dwpt16', 'rhum16', 'prcp16', 'wdir16', 'wspd16', 'prec24h16',
                    'days_since_rain', 'sum_consecutive_rainfall',
                    'sum_rain_last_7_days',
                    'sum_snow_last_7_days', 'snow24h', 'snow24h16',
                    'elevation',
                    'population',
                    #'sentinel',
                    'foret_encoder',
                    'argile_encoder',
                    #'id_encoder',
                    'cluster_encoder',
                    'cosia_encoder',
                    'vigicrues',
                    #'foret',
                    #'highway',
                    'cosia',
                    'Calendar',
                    #'Historical',
                    'Geo',
                    #'air',
                    'nappes',
                    #'AutoRegressionReg',
                    'AutoRegressionBin',
                ]

METHODS_SPATIAL_TRAIN = ['mean', 'sum', 'max', 'min']

def get_features_name_list(scale, features, methods):
    features_name = []
    if scale == 0:
        methods = ['mean']
    for var in features:
        if var == 'Calendar':
            features_name += calendar_variables
        elif var == 'air':
            features_name += air_variables
        elif var in landcover_variables:
            features_name += [f'{var}_{met}' for met in methods]
        elif var == 'sentinel':
            features_name += [f'{v}_{met}' for v in sentinel_variables for met in methods]
        elif var == "foret":
            features_name += [f'{v}_{met}' for v in foret_variables for met in methods]
        elif var == 'dynamicWorld':
            features_name += [f'{v}_{met}' for v in dynamic_world_variables for met in methods]
        elif var == 'cosia':
            features_name += [f'{v}_{met}' for v in cosia_variables for met in methods]
        elif var == 'highway':
            features_name += [f'{v}_{met}' for v in osmnx_variables for met in methods]
        elif var == 'Geo':
            features_name += geo_variables
        elif var == 'vigicrues':
            features_name += [f'{v}_{met}' for v in vigicrues_variables for met in methods]
        elif var == 'nappes':
            features_name += [f'{v}_{met}' for v in nappes_variables for met in methods]
        elif var == 'Historical':
            features_name += [f'{v}' for v in historical_variables]
        elif var == 'AutoRegressionReg':
            features_name += [f'AutoRegressionReg-{v}' for v in auto_regression_variable_reg]
        elif var == 'AutoRegressionBin':
            features_name +=  [f'AutoRegressionBin-{v}' for v in auto_regression_variable_bin]
        elif var == 'elevation':
            features_name += [f'{v}_{met}' for v in elevation_variables for met in methods]
        elif var == 'population':
            features_name += [f'{v}_{met}' for v in population_variabes for met in methods]
        elif var == 'region_class':
            features_name += [var]
        elif var == 'Past_risk':
            features_name += [var]
        elif var in varying_time_variables_name:
            features_name += [var]
        elif var == 'temporal_prediction' or var == 'spatial_prediction':
            features_name += [var]
        elif var.find('frequencyratio') != -1:
            features_name += [var]
        elif var == 'cluster_encoder':
            features_name += [var]
        else:
            features_name += [f'{var}_{met}' for met in methods]

    return features_name, len(features_name)

features_name, _ = get_features_name_list('departement', train_features, METHODS_SPATIAL_TRAIN)

In [ ]:
start_date = pd.Timestamp('2017-05-12')

data_full_dep['date'] = start_date + pd.to_timedelta(data_full_dep['date'], unit='D')

max_date = data_full_dep['date'].max()
print(f"📅 Dernière date obtenue : {max_date}")

print(data_full_dep[['date']].head())
print(data_full_dep[['date']].tail())

In [ ]:
data_firepoint['Date de première alerte'] = pd.to_datetime(data_firepoint['Date de première alerte'])
data_firepoint['Date de première alerte'] = data_firepoint['Date de première alerte'].dt.date
data_firepoint['Date de première alerte']

In [ ]:
data_full_dep['departement'] = data_full_dep['departement'].astype(int)
print(data_full_dep['departement'].unique())

In [ ]:
"""data_firepoint['Date de première alerte'] = pd.to_datetime(data_firepoint['Date de première alerte']).dt.strftime('%Y-%m-%d')
data_full_dep['date'] = pd.to_datetime(data_full_dep['date']).dt.strftime('%Y-%m-%d')

fire_counts = (
    data_firepoint.groupby(['Date de première alerte', 'Département', 'Nature'])
    .size()
    .reset_index(name='nombre_feux')
)

fire_counts.rename(columns={'Date de première alerte': 'date', 'Département': 'departement'}, inplace=True)

data_full_dep['departement'] = data_full_dep['departement'].astype(str)
fire_counts['departement'] = fire_counts['departement'].astype(str)

data_full_dep = data_full_dep.merge(
    fire_counts, on=['date', 'departement'], how='left'
)

pivoted_fire_counts = fire_counts.pivot_table(
    index=['date', 'departement'],
    columns='Nature',
    values='nombre_feux',
    aggfunc='sum',
    fill_value=0
)

data_full_dep = data_full_dep.merge(
    pivoted_fire_counts,
    on=['date', 'departement'],
    how='left'
)

data_full_dep.fillna(0, inplace=True)"""

---
#### 3.2 Modélisation

---
##### 3.2.1 Proba des feux

In [ ]:
#data_filtered = data_full_dep[(data_full_dep['nombre_feux'] > 0) & (data_full_dep['Nature'].notna())].copy()

In [ ]:
#data_filtered.to_csv("../data/data_filtered.csv", index=False)

In [ ]:
data_filtered = pd.read_csv("../data/data_filtered.csv")

In [ ]:
data_filtered.shape

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

data_filtered['date'] = pd.to_datetime(data_filtered['date'], errors='coerce')

data_filtered['y_acc'] = (data_filtered['Nature'] == 'Accidentelle').astype(int)
data_filtered['y_mal'] = (data_filtered['Nature'] == 'Malveillance').astype(int)
data_filtered['y_inv_par'] = (data_filtered['Nature'] == 'Involontaire (particulier)').astype(int)
data_filtered['y_inv_trav'] = (data_filtered['Nature'] == 'Involontaire (travaux)').astype(int)
data_filtered['y_nat'] = (data_filtered['Nature'] == 'Naturelle').astype(int)

X = data_filtered[features_name].fillna(0)

y_dict = {
    'Accidentelle': data_filtered['y_acc'],
    'Malveillance': data_filtered['y_mal'],
    'Involontaire (particulier)': data_filtered['y_inv_par'],
    'Involontaire (travaux)': data_filtered['y_inv_trav'],
    'Naturelle': data_filtered['y_nat']
}

models = {
    "RandomForest": RandomForestClassifier(random_state=42, class_weight='balanced'),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42, auto_class_weights='Balanced'),
    "LightGBM": LGBMClassifier(verbose=-1, random_state=42, class_weight='balanced'),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', verbose=0, random_state=42, scale_pos_weight=1)
}

def apply_smote(X_train, y_train, label):
    unique_classes = np.unique(y_train)
    print(f"🔹 Distribution des classes pour {label} avant SMOTE : {np.bincount(y_train)}")

    if len(unique_classes) > 1:
        smote = SMOTE(random_state=42)
        X_res, y_res = smote.fit_resample(X_train, y_train)
        print(f"✅ SMOTE appliqué à {label}, nouvelle distribution : {np.bincount(y_res)}")
        return X_res, y_res
    else:
        print(f"⚠️ Pas assez de classes pour appliquer SMOTE à {label}, pas de suréchantillonnage.")
        return X_train, y_train

y_pred = {model_name: {} for model_name in models}
metrics = {model_name: {} for model_name in models}

train_data = data_filtered[data_filtered['date'] <= '2022-12-31']
test_data = data_filtered[data_filtered['date'] >= '2023-01-01']

print(f"Train data count: {len(train_data)}")
print(f"Test data count: {len(test_data)}")

target_names = {
    'Accidentelle': 'y_acc',
    'Malveillance': 'y_mal',
    'Involontaire (particulier)': 'y_inv_par',
    'Involontaire (travaux)': 'y_inv_trav',
    'Naturelle': 'y_nat'
}

for label, y in y_dict.items():
    print(f"\n📌 **Entraînement pour {label}**")

    X_train = train_data[features_name].fillna(0)
    y_train = train_data[target_names[label]]
    X_test = test_data[features_name].fillna(0)
    y_test = test_data[target_names[label]]

    X_train_res, y_train_res = apply_smote(X_train, y_train, label)

    for model_name, model in models.items():
        model.fit(X_train_res, y_train_res)

        if len(model.classes_) > 1:
            y_pred_proba = model.predict_proba(X_test)[:, 1]
            y_pred_classes = model.predict(X_test)
        else:
            print(f"⚠️ Modèle {model_name} pour {label} n'a qu'une seule classe, retour de 0.")
            y_pred_proba = np.zeros(X_test.shape[0])
            y_pred_classes = np.zeros(X_test.shape[0])

        metrics[model_name][label] = {
            "Accuracy": accuracy_score(y_test, y_pred_classes),
            "Precision": precision_score(y_test, y_pred_classes, zero_division=0),
            "Recall": recall_score(y_test, y_pred_classes, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred_classes, zero_division=0)
        }

        y_pred[model_name][label] = y_pred_proba

results_list = []

for model_name, model_metrics in metrics.items():
    for label, metric_values in model_metrics.items():
        results_list.append({
            "Modèle": model_name,
            "Nature": label,
            "Accuracy": metric_values["Accuracy"],
            "Precision": metric_values["Precision"],
            "Recall": metric_values["Recall"],
            "F1 Score": metric_values["F1 Score"]
        })

df_results = pd.DataFrame(results_list)
df_results.to_csv("../results_img_csv/resultats_moyenne_tous_modeles.csv", index=False)

print("\n✅ Résultats enregistrés dans 'resultats_moyenne_tous_modeles.csv'")

for model_name, metrics_for_model in metrics.items():
    print(f"\n🔹 **Métriques pour le modèle {model_name}**")
    for label, metric_values in metrics_for_model.items():
        print(f"\n📊 **{label}**")
        for metric_name, metric_value in metric_values.items():
            print(f"{metric_name}: {metric_value:.4f}")

In [ ]:
df_results_pred = pd.read_csv("../results_img_csv/resultats_moyenne_tous_modeles.csv")
df_results_pred

---
##### 3.2.2 Autres features

---
Cluster Encoder

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model_results = {}

clusters = data_filtered["cluster_encoder"].unique()

for cluster in clusters:
    print(f"\n🔹 **Modèle pour Cluster {cluster}**")

    data_cluster = data_filtered[data_filtered["cluster_encoder"] == cluster]
    X_cluster = data_cluster[X.columns]
    y_cluster = data_cluster["Nature"]
    class_counts = y_cluster.value_counts()
    print(f"📊 Distribution des classes dans ce cluster :\n{class_counts}\n")

    if len(X_cluster) < 10:
        print(f"⚠️ Trop peu de données ({len(X_cluster)}) pour entraîner un modèle sur le cluster {cluster}. On passe.")
        continue

    min_class_size = class_counts.min()

    if min_class_size < 2:
        print(f"⚠️ Certaines classes ont trop peu d'échantillons. Pas de stratification utilisée.")
        stratify_option = None
    else:
        stratify_option = y_cluster

    X_train, X_test, y_train, y_test = train_test_split(X_cluster, y_cluster, test_size=0.2, random_state=42, stratify=stratify_option)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    model_results[cluster] = {
        "Modèle": model,
        "Accuracy": accuracy,
        "Rapport": report
    }

    print(f"✅ Précision du modèle : {accuracy:.4f}")
    print(report)

---
##### 3.2.3 Département par département

In [ ]:
feux_par_departement = data_full_dep["departement"].value_counts().reset_index()
feux_par_departement.columns = ["Département", "Nombre de Feux"]

print(feux_par_departement.sort_values(by='Département'))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

data_filtered['y_acc'] = (data_filtered['Nature'] == 'Accidentelle').astype(int)
data_filtered['y_mal'] = (data_filtered['Nature'] == 'Malveillance').astype(int)
data_filtered['y_inv_par'] = (data_filtered['Nature'] == 'Involontaire (particulier)').astype(int)
data_filtered['y_inv_trav'] = (data_filtered['Nature'] == 'Involontaire (travaux)').astype(int)
data_filtered['y_nat'] = (data_filtered['Nature'] == 'Naturelle').astype(int)

X = data_filtered[features_name].fillna(0)

y_dict = {
    'Accidentelle': data_filtered['y_acc'],
    'Malveillance': data_filtered['y_mal'],
    'Involontaire (particulier)': data_filtered['y_inv_par'],
    'Involontaire (travaux)': data_filtered['y_inv_trav'],
    'Naturelle': data_filtered['y_nat']
}

rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')

def apply_smote(X_train, y_train):
    unique_classes, class_counts = np.unique(y_train, return_counts=True)

    if len(unique_classes) < 2:
        print("⚠️ Impossible d'appliquer SMOTE : une seule classe présente.")
        return X_train, y_train

    minority_class_size = min(class_counts)
    if minority_class_size < 2:
        print(f"⚠️ Trop peu d'échantillons ({minority_class_size}) pour SMOTE, pas d'oversampling.")
        return X_train, y_train
    n_neighbors = min(2, minority_class_size - 1)

    print(f"✅ SMOTE appliqué avec n_neighbors={n_neighbors} (classe minoritaire : {minority_class_size} échantillons)")
    smote = SMOTE(random_state=42, k_neighbors=n_neighbors)
    return smote.fit_resample(X_train, y_train)

results_list = []


for department in data_filtered["departement"].unique():

    print(f"\n📌 **Traitement du département {department}**")

    data_dep = data_filtered[data_filtered["departement"] == department]
    X_dep = data_dep[features_name].fillna(0)

    if len(data_dep) < 10:
        print(f"⚠️ Département {department} a trop peu de données ({len(data_dep)}), on passe.")
        continue

    for label, y in y_dict.items():
        y_dep = y.loc[data_dep.index]

        if y_dep.sum() == 0:
            print(f"⚠️ Pas de feux '{label}' dans le département {department}, on passe.")
            continue

        unique_classes, class_counts = np.unique(y_dep, return_counts=True)
        if len(unique_classes) < 2:
            print(f"⚠️ Département {department} - '{label}' a une seule classe, on passe.")
            continue

        if class_counts.min() < 2:
            print(f"⚠️ Département {department} - '{label}' a une classe avec moins de 2 observations, désactivation de `stratify`.")
            stratify_param = None
        else:
            stratify_param = y_dep

        X_train, X_test, y_train, y_test = train_test_split(X_dep, y_dep, test_size=0.2, random_state=42, stratify=stratify_param)
        X_train_res, y_train_res = apply_smote(X_train, y_train)
        rf_model.fit(X_train_res, y_train_res)
        y_pred_classes = rf_model.predict(X_test)

        metrics = {
            "Département": department,
            "Type de feu": label,
            "Accuracy": accuracy_score(y_test, y_pred_classes),
            "Precision": precision_score(y_test, y_pred_classes, zero_division=0),
            "Recall": recall_score(y_test, y_pred_classes, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred_classes, zero_division=0)
        }

        results_list.append(metrics)

df_results = pd.DataFrame(results_list)
df_results.to_csv("../results_img_csv/resultats_random_forest_par_departement.csv", index=False)
print("\n Résultats enregistrés dans 'resultats_random_forest_par_departement.csv'")

---
#### 3.4 Méthodes avancées

In [ ]:
import pandas as pd

data_filtered['date'] = pd.to_datetime(data_filtered['date'])
data_filtered['mois'] = data_filtered['date'].dt.month

def assign_saison(mois):
    if mois in [12, 1, 2]:
        return 'Hiver'
    elif mois in [3, 4, 5]:
        return 'Printemps'
    elif mois in [6, 7, 8]:
        return 'Été'
    else:
        return 'Automne'

data_filtered['Saison'] = data_filtered['mois'].apply(assign_saison)
print(data_filtered[['date', 'Saison']].head())

---
##### 3.4.1 Under sampling

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
import pandas as pd
import numpy as np

def apply_undersampling(X_train, y_train, X_test, y_test, label, fraction=0.3):
    """
    Réduit la classe majoritaire dans X_train/y_train, déplace les données exclues dans X_test/y_test.
    """
    df = pd.concat([X_train, y_train], axis=1)
    class_counts = y_train.value_counts()

    if len(class_counts) < 2:
        print(f"⚠️ {label}: Pas assez de classes.")
        return X_train, y_train, X_test, y_test

    majority_class = class_counts.idxmax()
    minority_class = class_counts.idxmin()

    df_majority = df[df[y_train.name] == majority_class]
    df_minority = df[df[y_train.name] == minority_class]

    num_to_keep = int(len(df_majority) * fraction)
    df_majority_kept = df_majority.sample(num_to_keep, random_state=42)
    df_majority_removed = df_majority.drop(df_majority_kept.index)

    df_resampled = pd.concat([df_majority_kept, df_minority], axis=0).sample(frac=1, random_state=42)
    X_resampled = df_resampled.drop(columns=y_train.name)
    y_resampled = df_resampled[y_train.name]

    X_test_updated = pd.concat([X_test, df_majority_removed.drop(columns=y_train.name)], axis=0).reset_index(drop=True)
    y_test_updated = pd.concat([y_test, df_majority_removed[y_train.name]], axis=0).reset_index(drop=True)

    return X_resampled, y_resampled, X_test_updated, y_test_updated

X_train_resampled_dict = {}
y_train_resampled_dict = {}
X_test_updated_dict = {}
y_test_updated_dict = {}

for label, y in y_dict.items():
    X_train = train_data[features_name].fillna(0)
    y_train = train_data[target_names[label]]
    X_test = test_data[features_name].fillna(0)
    y_test = test_data[target_names[label]]

    print(f"\n🔍 Label : {label}")
    print("Avant under-sampling (train) :", np.bincount(y_train))
    print("Avant under-sampling (test)  :", np.bincount(y_test))

    X_train_res, y_train_res, X_test_upd, y_test_upd = apply_undersampling(X_train, y_train, X_test, y_test, label, fraction=0.8)

    X_train_resampled_dict[label] = X_train_res
    y_train_resampled_dict[label] = y_train_res
    X_test_updated_dict[label] = X_test_upd
    y_test_updated_dict[label] = y_test_upd

    print("Après under-sampling (train):", np.bincount(y_train_res))
    print("Après under-sampling (test) :", np.bincount(y_test_upd))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
metrics = {}
y_pred = {}

for label in y_train_resampled_dict.keys():
    print(f"\n🔹 **Évaluation pour '{label}'**")

    X_train_res = X_train_resampled_dict[label]
    y_train_res = y_train_resampled_dict[label]
    X_test_upd = X_test_updated_dict[label]
    y_test_upd = y_test_updated_dict[label]

    if len(np.unique(y_test_upd)) > 1:
        model.fit(X_train_res, y_train_res)
        y_pred_classes = model.predict(X_test_upd)
    else:
        print(f"⚠️ '{label}' contient une seule classe en test. Prédiction constante.")
        y_pred_classes = np.full_like(y_test_upd, fill_value=np.unique(y_test_upd)[0])

    metrics[label] = {
        "Accuracy": accuracy_score(y_test_upd, y_pred_classes),
        "Precision": precision_score(y_test_upd, y_pred_classes, zero_division=0),
        "Recall": recall_score(y_test_upd, y_pred_classes, zero_division=0),
        "F1 Score": f1_score(y_test_upd, y_pred_classes, zero_division=0)
    }

    y_pred[label] = y_pred_classes

df_results = pd.DataFrame([
    {
        "Modèle": "RandomForest",
        "Nature": label,
        **scores
    }
    for label, scores in metrics.items()
])

df_results.to_csv("../results_img_csv/resultats_undersampling_randomforest.csv", index=False)
print("\n✅ Résultats enregistrés dans 'resultats_undersampling_randomforest.csv'")
df_results

---
##### 3.4.2 Multi output classifier

In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE

y_train = train_data[['y_acc', 'y_mal', 'y_inv_par', 'y_inv_trav', 'y_nat']]
y_test = test_data[['y_acc', 'y_mal', 'y_inv_par', 'y_inv_trav', 'y_nat']]

if 'date' in X_train.columns:
    print("📅 Conversion de la colonne 'date' en format numérique...")
    X_train['date'] = pd.to_datetime(X_train['date']).astype(int) / 10**9  # Convertir en secondes
    X_test['date'] = pd.to_datetime(X_test['date']).astype(int) / 10**9
    print("✅ Conversion terminée.")

smote = SMOTE(random_state=42)

X_train_resampled = None
y_train_resampled = pd.DataFrame()

for label in y_train.columns:
    print(f"🔄 Application de SMOTE pour {label}...")

    X_res, y_res = smote.fit_resample(X_train, y_train[label])

    if X_train_resampled is None:
        X_train_resampled = X_res

    y_train_resampled[label] = y_res

y_train_resampled = y_train_resampled.fillna(0)

print("✅ SMOTE appliqué, les classes sont maintenant équilibrées.")

base_model = RandomForestClassifier(random_state=42, class_weight="balanced")

multi_target_model = MultiOutputClassifier(base_model)

multi_target_model.fit(X_train_resampled, y_train_resampled)

y_pred = multi_target_model.predict(X_test)

metrics = {}
for i, label in enumerate(y_train.columns):
    accuracy = accuracy_score(y_test.iloc[:, i], y_pred[:, i])
    precision = precision_score(y_test.iloc[:, i], y_pred[:, i], zero_division=0)
    recall = recall_score(y_test.iloc[:, i], y_pred[:, i], zero_division=0)
    f1 = f1_score(y_test.iloc[:, i], y_pred[:, i], zero_division=0)

    metrics[label] = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    }

for label, metric_values in metrics.items():
    print(f"\n📊 **{label}**")
    for metric_name, metric_value in metric_values.items():
        print(f"{metric_name}: {metric_value:.4f}")

---
##### 3.4.3 Techniques d'ensemble (vote / bagging / stacking)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Vérification si y_train et y_test sont dans le format one-hot encoding (dans ce cas, conversion en labels numériques)
if len(y_train.shape) > 1 and y_train.shape[1] > 1:
    y_train = np.argmax(y_train, axis=1)
    y_test = np.argmax(y_test, axis=1)

# Définir les modèles de base
base_models = [
    ('rf', RandomForestClassifier(n_estimators=250, max_depth=4, random_state=42, class_weight='balanced', n_jobs=-1)),
    ('xgb', XGBClassifier(n_estimators=100, learning_rate=0.07, max_depth=2, scale_pos_weight=3, use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)),
    ('lgbm', LGBMClassifier(n_estimators=50, learning_rate=0.06, max_depth=5, class_weight='balanced', random_state=42, n_jobs=-1)),
]

# Créer le modèle VotingClassifier
voting_clf = VotingClassifier(estimators=base_models, voting='hard', n_jobs=-1)

# Entraîner le modèle VotingClassifier
voting_clf.fit(X_train, y_train)

# Faire des prédictions sur le jeu de test
y_pred = voting_clf.predict(X_test)

# Calculer les métriques globales
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')  # Utilisation de la moyenne pondérée pour la multi-classe
recall = recall_score(y_test, y_pred, average='weighted')  # Même chose pour recall
f1 = f1_score(y_test, y_pred, average='weighted')  # Idem pour F1-score

# Afficher les résultats globaux
print(f"\n🔹 **Performance du modèle Voting Classifier sur le Test Set :**")
print(f"✅ Accuracy: {accuracy:.4f}")
print(f"✅ Precision: {precision:.4f}")
print(f"✅ Recall: {recall:.4f}")
print(f"✅ F1-Score: {f1:.4f}")

# Afficher les métriques détaillées pour chaque classe avec les noms des classes
print("\n🔹 **Métriques détaillées pour chaque classe :**")
print(classification_report(y_test, y_pred))  # Convertir les classes en chaînes si nécessaire

# 📌 Création d'un DataFrame pour stocker les métriques globales
metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Value": [accuracy, precision, recall, f1]
})

# 💾 Sauvegarde des résultats globaux dans un CSV
metrics_df.to_csv("../results_img_csv/voting_classifier_results_global.csv", index=False)
print("\n✅ Résultats globaux enregistrés dans 'voting_classifier_results_global.csv'")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
#from catboost import CatBoostClassifier
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Si nécessaire, convertir les labels en un seul vecteur (si c'est du one-hot encoding)
if len(y_train.shape) > 1 and y_train.shape[1] > 1:
    y_train = np.argmax(y_train, axis=1)
    y_test = np.argmax(y_test, axis=1)

base_models = [
    ('rf', RandomForestClassifier(n_estimators=250, max_depth=4, random_state=42, class_weight='balanced', n_jobs=-1)),
    ('xgb', XGBClassifier(n_estimators=100, learning_rate=0.07, max_depth=2, scale_pos_weight=3, use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)),
    ('cat', CatBoostClassifier(iterations=100, learning_rate=0.1, depth=5, auto_class_weights='Balanced', random_seed=42, verbose=0, early_stopping_rounds=10)),
    ('lgbm', LGBMClassifier(n_estimators=50, learning_rate=0.06, max_depth=5, class_weight='balanced', random_state=42, n_jobs=-1)),
]

# Standardisation des données
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Créer un BaggingClassifier avec chaque modèle de base
bagging_clfs = []

# Création des BaggingClassifiers pour chaque modèle de base
for model in base_models:
    bagging_clf = BaggingClassifier(
        estimator=RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
        n_estimators=10,
        max_samples=0.8,
        max_features=0.8,
        bootstrap=True,
        n_jobs=-1,
    random_state=42
)
    bagging_clfs.append(bagging_clf)

# Entraîner les BaggingClassifiers sur les données d'entraînement et obtenir les prédictions
predictions = []
for bagging_clf in bagging_clfs:
    bagging_clf.fit(X_train_scaled, y_train)
    predictions.append(bagging_clf.predict(X_test_scaled))

# Moyenne des prédictions de tous les modèles
final_predictions = np.mean(predictions, axis=0)
final_predictions = np.round(final_predictions).astype(int)

# Calculer les métriques de performance
accuracy = accuracy_score(y_test, final_predictions)
precision = precision_score(y_test, final_predictions, average='weighted')  # Utilisation de la moyenne pondérée pour la multi-classe
recall = recall_score(y_test, final_predictions, average='weighted')  # Même chose pour recall
f1 = f1_score(y_test, final_predictions, average='weighted')  # Idem pour F1-score

# Afficher les résultats
print(f"\n🔹 **Performance du modèle Bagging avec les Base Models sur le Test Set :**")
print(f"✅ Accuracy: {accuracy:.4f}")
print(f"✅ Precision: {precision:.4f}")
print(f"✅ Recall: {recall:.4f}")
print(f"✅ F1-Score: {f1:.4f}")

# Afficher les métriques détaillées pour chaque classe avec les noms des classes
print("\n🔹 **Métriques détaillées pour chaque classe :**")
print(classification_report(y_test, y_pred))  # Convertir les classes en chaînes si nécessaire

# 📌 Création d'un DataFrame pour stocker les métriques globales
metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Value": [accuracy, precision, recall, f1]
})

# 💾 Sauvegarde des résultats globaux dans un CSV
metrics_df.to_csv("../results_img_csv/bagging_classifier_results_global.csv", index=False)
print("\n✅ Résultats globaux enregistrés dans 'bagging_classifier_results_global.csv'")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
#from catboost import CatBoostClassifier
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression

# Si nécessaire, convertir les labels en un seul vecteur (si c'est du one-hot encoding)
if len(y_train.shape) > 1 and y_train.shape[1] > 1:
    y_train = np.argmax(y_train, axis=1)
    y_test = np.argmax(y_test, axis=1)

# Définir les modèles de base
base_models = [
    ('rf', RandomForestClassifier(n_estimators=500, max_depth=4, random_state=42, class_weight='balanced', n_jobs=-1)),
    ('xgb', XGBClassifier(n_estimators=100, learning_rate=0.07, max_depth=2, scale_pos_weight=2, use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)),
    ('cat', CatBoostClassifier(iterations=100, learning_rate=0.1, depth=5, auto_class_weights='Balanced', random_seed=42, verbose=0, early_stopping_rounds=10)),
    ('lgbm', LGBMClassifier(n_estimators=50, learning_rate=0.06, max_depth=5, class_weight='balanced', random_state=42, n_jobs=-1)),
]

meta_model = RandomForestClassifier(n_estimators=250, random_state=42, class_weight='balanced')

# Standardisation des données avec RobustScaler (alternative à StandardScaler)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Rééchantillonnage des données d'entraînement avec SMOTE si les données sont déséquilibrées
#smote = SMOTE(random_state=42)
#X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# Créer un StackingClassifier avec les modèles de base et un métamodèle
stacking_clf = StackingClassifier(estimators=base_models, final_estimator=meta_model, n_jobs=-1)

#X_train = X_train_resampled
#y_train = y_train_resampled

# Entraîner le StackingClassifier sur les données d'entraînement rééchantillonnées
stacking_clf.fit(X_train_scaled, y_train)
y_pred = stacking_clf.predict(X_test_scaled)

# Calculer les métriques de performance
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')  # Utilisation de la moyenne pondérée pour la multi-classe
recall = recall_score(y_test, y_pred, average='weighted')  # Même chose pour recall
f1 = f1_score(y_test, y_pred, average='weighted')  # Idem pour F1-score

# Afficher les résultats
print(f"\n🔹 **Performance du modèle Stacking avec les Base Models sur le Test Set :**")
print(f"✅ Accuracy: {accuracy:.4f}")
print(f"✅ Precision: {precision:.4f}")
print(f"✅ Recall: {recall:.4f}")
print(f"✅ F1-Score: {f1:.4f}")

# Afficher les métriques détaillées pour chaque classe avec les noms des classes
class_names = np.unique(y_train)  # Récupérer les noms des classes depuis y_train
print("\n🔹 **Métriques détaillées pour chaque classe :**")
print(classification_report(y_test, y_pred, target_names=class_names.astype(str)))  # Affichage des classes avec leurs noms

# 📌 Création d'un DataFrame pour stocker les métriques globales
metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Value": [accuracy, precision, recall, f1]
})

# 💾 Sauvegarde des résultats globaux dans un CSV
metrics_df.to_csv("../results_img_csv/stacking_classifier_results_global.csv", index=False)
print("\n✅ Résultats globaux enregistrés dans 'stacking_classifier_results_global.csv'")

---
##### 3.4.4 Réseaux de neurones

Multi Layer Perceptron (MLP)

In [ ]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 🚀 Vérification du format des labels (si one-hot encoding, on les transforme en vecteur)
if len(y_train.shape) > 1 and y_train.shape[1] > 1:
    y_train = np.argmax(y_train, axis=1)
    y_test = np.argmax(y_test, axis=1)

# Vérifier la forme de X_train et X_test
print("Forme de X_train:", X_train.shape)
print("Forme de X_test:", X_test.shape)

# 🔄 Normalisation avec StandardScaler
scaler = StandardScaler()

# Vérification que X_train et X_test sont bien en format 2D avant la normalisation
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Vérifier la forme après normalisation
print("Forme de X_train_scaled:", X_train_scaled.shape)
print("Forme de X_test_scaled:", X_test_scaled.shape)

# ⚖️ SMOTE pour rééquilibrer les classes
smote = SMOTE(sampling_strategy='auto', random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# 🧠 Définition du MLP avec plus de couches cachées
mlp = MLPClassifier(hidden_layer_sizes=(100, 100, 100, 100, 100, 100, 100),
                    activation='relu',
                    solver='adam',
                    alpha=0.001,
                    learning_rate_init=0.01,
                    max_iter=500000,
                    batch_size=128,
                    early_stopping=True,
                    validation_fraction=0.1,
                    random_state=42)

# 🎯 Entraînement du modèle
mlp.fit(X_train_resampled, y_train_resampled)

# 🔮 Prédictions
y_pred_proba = mlp.predict_proba(X_test_scaled)  # Probabilités pour chaque classe
y_pred = np.argmax(y_pred_proba, axis=1)  # Classe avec la plus haute probabilité

# 📊 Calcul des métriques pour classification multiclasse
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average='weighted'),
    "Recall": recall_score(y_test, y_pred, average='weighted'),
    "F1-Score": f1_score(y_test, y_pred, average='weighted')
}

# 📌 Affichage des résultats
metrics_df = pd.DataFrame(metrics, index=["Valeurs"]).T
print("\n🔹 **Performance du MLP sur le Test Set :**")
print(metrics_df)

# 📊 Affichage du rapport détaillé
print("\n🔹 **Métriques détaillées pour chaque classe :**")
print(classification_report(y_test, y_pred))

metrics_df.to_csv("../results_img_csv/MLP_results.csv", index=False)

---
##### 3.4.5 SHAP

In [ ]:
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# 🔢 X et y
X = data_filtered[features_name].fillna(0)
y = data_filtered['Nature']  # Catégories texte

# 🎯 Encodage des classes (texte → int)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 🔁 Standardisation
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# ✂️ Split stratifié train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

# 🧠 Modèle XGBoost
model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

# ✅ Initialisation de SHAP
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

# 📌 Classe à visualiser
selected_class_name = "Naturelle"
selected_class_idx = list(le.classes_).index(selected_class_name)

print(f"✅ Visualisation des SHAP values pour la classe : {selected_class_name} (index: {selected_class_idx})")

# 📊 Visualisation SHAP
plt.figure()
shap.summary_plot(
    shap_values[:, :, selected_class_idx],
    X_test,
    feature_names=features_name,
    show=False  # Ne pas afficher pour pouvoir sauvegarder
)

# 💾 Enregistrement dans le dossier "resultats"
output_path = "../results_img_csv"
filename = f"shap_summary_{selected_class_name.replace(' ', '_')}.png"
plt.savefig(os.path.join(output_path, filename), bbox_inches="tight", dpi=300)
plt.close()
print(f"✅ SHAP summary plot enregistré sous : {os.path.join(output_path, filename)}")

---
##### 3.4.6 Explicabilité avec XGB

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split
import os

# 🔢 Données d’entrée
X = data_filtered[features_name].fillna(0)
y = data_filtered['Nature']

# 🧠 Encodage des labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 🔁 Standardisation
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# ✂️ Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

# 📌 Entraînement du modèle XGBoost
model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

# 📊 Récupération des importances
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

# 🧾 Affichage top features
top_n = 20
top_features = [(features_name[i], importances[i]) for i in indices[:top_n]]

print("🔍 Top 20 features les plus importantes :")
for i, (feat, score) in enumerate(top_features):
    print(f"{i+1:>2}. {feat:<30} {score:.4f}")

# 📈 Visualisation
plt.figure(figsize=(10, 6))
plt.barh([features_name[i] for i in indices[:top_n]][::-1],
         [importances[i] for i in indices[:top_n]][::-1],
         color='teal')
plt.title("Top 20 des variables importantes (XGBoost)")
plt.xlabel("Importance")
plt.tight_layout()

# 💾 Sauvegarde
output_path = "../results_img_csv"
os.makedirs(output_path, exist_ok=True)
plt.savefig(os.path.join(output_path, "xgboost_feature_importance.png"), dpi=300)
plt.close()

---
#### 3.5 Clustering

---
##### 3.5.0 Variances avec PCA (pour le nombre de composantes)

---
##### 3.5.1 KMeans

---
###### 3.5.1.1 PCA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import MinMaxScaler

# 📌 Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 📉 Réduction dimensionnelle avec PCA
pca = PCA(n_components=47)
X_pca = pca.fit_transform(X_scaled)

# 📊 Initialisation des listes
sil_scores = []
db_scores = []

# 🔁 Boucle sur différents k
for k in range(1, 26):
    kmeans = KMeans(n_clusters=k, init='k-means++', max_iter=300, n_init=10, random_state=42)
    kmeans.fit(X_pca)

    if k > 1:
        sil_scores.append(silhouette_score(X_pca, kmeans.labels_))
        db_scores.append(davies_bouldin_score(X_pca, kmeans.labels_))

# 📈 Silhouette Score
plt.figure(figsize=(10, 6))
plt.plot(range(2, 26), sil_scores, marker='o', linestyle='--', color='green')
plt.title('📊 Silhouette Score par nombre de clusters')
plt.xlabel('Nombre de clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

# 📈 Davies-Bouldin Index
plt.figure(figsize=(10, 6))
plt.plot(range(2, 26), db_scores, marker='o', linestyle='--', color='orange')
plt.title('📉 Davies-Bouldin Index par nombre de clusters')
plt.xlabel('Nombre de clusters')
plt.ylabel("Davies-Bouldin Index (↓ mieux)")
plt.grid(True)
plt.show()

# 🔍 Nombre optimal basé sur Silhouette
optimal_k = sil_scores.index(max(sil_scores)) + 2
print(f"✅ Nombre optimal de clusters selon le Silhouette Score : {optimal_k}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans

# Appliquer KMeans sur X_pca (qui peut avoir 47 dimensions)
kmeans = KMeans(n_clusters=24, init='k-means++', max_iter=300, n_init=10, random_state=42)
kmeans.fit(X_pca)

# Récupérer les labels et les centres
kmeans_PCA_labels = kmeans.labels_
centroids = kmeans.cluster_centers_
X_pca_visu = X_pca[:, :2]
centroids_2D = centroids[:, :2]
# Visualisation 2D des clusters
plt.figure(figsize=(10, 8))
plt.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=kmeans_PCA_labels, cmap='plasma',
    s=50, marker='o', edgecolors='k', alpha=0.6
)

# Tracer les centres des clusters
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, marker='X', label='Centres des clusters')

# Final touches
plt.title(f'Clustering KMeans avec {17} clusters (affichage 2D)')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import folium
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.cm as cm

# Assure-toi que `kmeans_PCA_labels` est déjà défini
# Ici, kmeans_PCA_labels = kmeans.labels_

# Créer une carte centrée sur la France
map_france = folium.Map(location=[46.603354, 1.888334], zoom_start=6)  # Coordonnées géographiques approximatives de la France

# Générer 16 couleurs distinctes en utilisant matplotlib
colors = cm.get_cmap('tab20', 24)  # 'tab20' génère 20 couleurs distinctes
cluster_colors = [colors(i) for i in range(24)]  # Attribuer une couleur à chaque cluster

# Afficher les points des clusters sur la carte
for i, row in data_filtered.iterrows():
    # Limiter les valeurs des coordonnées pour s'assurer qu'elles sont dans les bornes de la France
    latitude = np.clip(row['latitude'], 42, 51)
    longitude = np.clip(row['longitude'], -5, 10)

    # Utiliser kmeans_PCA_labels pour récupérer le numéro du cluster pour chaque point
    cluster_label = kmeans_PCA_labels[i]

    # Attribuer une couleur en fonction du cluster
    cluster_color = cluster_colors[cluster_label]

    # Créer un popup avec le numéro du cluster
    popup = folium.Popup(f'Cluster: {cluster_label}', parse_html=True)

    folium.CircleMarker(
        location=[latitude, longitude],
        radius=5,
        color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill=True,
        fill_color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill_opacity=0.6,
        popup=popup  # Ajouter le popup ici
    ).add_to(map_france)

# Afficher la carte dans un fichier HTML
map_france.save('../results_img_csv/map_clusters_kmeans_PCA.html')

# Afficher un message de succès
print("✅ Carte avec les points des clusters et leurs numéros enregistrée sous 'map_clusters_kmeans_PCA.html'.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from scipy.spatial import ConvexHull

# Dictionnaire des couleurs des saisons
saison_colors = {
    'hiver': '#1f77b4',
    'printemps': '#2ca02c',
    'été': '#ff7f0e',
    'automne': '#d62728'
}

# Appliquer KMeans
kmeans = KMeans(n_clusters=24, init='k-means++', max_iter=300, n_init=10, random_state=42)
kmeans.fit(X_pca)

# Récupérer les labels et les centres
kmeans_PCA_labels = kmeans.labels_
centroids = kmeans.cluster_centers_
X_pca_visu = X_pca[:, :2]
centroids_2D = centroids[:, :2]
# Récupérer les saisons en minuscules
saisons = data_filtered["Saison"].str.lower().values

# Visualisation
plt.figure(figsize=(10, 8))

for cluster_id in range(24):
    cluster_mask = (kmeans_PCA_labels == cluster_id)
    cluster_points = X_pca_visu[cluster_mask]
    cluster_saisons = saisons[cluster_mask]

    # Points colorés par saison dans le cluster
    for saison in np.unique(cluster_saisons):
        saison_mask = (cluster_saisons == saison)
        plt.scatter(
            cluster_points[saison_mask, 0],
            cluster_points[saison_mask, 1],
            color=saison_colors.get(saison, '#999999'),
            label=f'Cluster {cluster_id} - {saison.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # 🔢 Affichage du numéro du cluster au centre
    plt.text(
        centroids_2D[cluster_id, 0], centroids_2D[cluster_id, 1],
        str(cluster_id), fontsize=10, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# Finalisation du plot
plt.title(f'Clustering KMeans (PCA) avec {optimal_k} clusters, colorés par saison')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from scipy.spatial import ConvexHull
import unidecode

# 🎨 Couleurs pour chaque nature
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 🎯 Appliquer KMeans
kmeans = KMeans(n_clusters=24, init='k-means++', max_iter=300, n_init=10, random_state=42)
kmeans.fit(X_pca)

# 🎯 Récupération des clusters et centres
kmeans_PCA_labels = kmeans.labels_
centroids = kmeans.cluster_centers_

X_pca_visu = X_pca[:, :2]
centroids_2D = centroids[:, :2]
# 🔤 Nettoyage et normalisation des natures
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

natures_raw = data_filtered["Nature"].str.strip().str.lower()
natures = natures_raw.apply(normalize_nature)
natures = natures.replace({
    'accidentel': 'accidentelle',
    'involontaire(particulier)': 'involontaire (particulier)',
    'involontaire ( particulier )': 'involontaire (particulier)',
    'involontaire(travaux)': 'involontaire (travaux)',
    'malveillance': 'malveillance',
    'naturelle': 'naturelle'
})

# 📊 Visualisation
plt.figure(figsize=(10, 8))

for cluster_id in range(24):
    cluster_mask = (kmeans_PCA_labels == cluster_id)
    cluster_points = X_pca_visu[cluster_mask]
    cluster_natures = natures[cluster_mask]

    # Coloration des points par nature
    for nature in np.unique(cluster_natures):
        nature_mask = (cluster_natures == nature)
        plt.scatter(
            cluster_points[nature_mask, 0],
            cluster_points[nature_mask, 1],
            color=nature_colors.get(nature, '#999999'),
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # 🔢 Affichage du numéro du cluster
    plt.text(
        centroids_2D[cluster_id, 0], centroids_2D[cluster_id, 1],
        str(cluster_id), fontsize=14, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# ➕ Légende simplifiée (1 par nature)
handles = [
    plt.Line2D([0], [0], marker='o', color='w', label=nature.capitalize(),
               markerfacecolor=color, markeredgecolor='k', markersize=10)
    for nature, color in nature_colors.items()
]

plt.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left')

# Finalisation du plot
plt.title(f'Clustering KMeans (PCA) avec {optimal_k} clusters, colorés par nature')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Créer un DataFrame avec les labels de KMeans et les saisons
data_for_test = pd.DataFrame({
    'Cluster': kmeans_PCA_labels,  # Les labels des clusters KMeans
    'Saison': data_filtered['Saison'].str.lower()  # Les saisons (en minuscule)
})

# Créer une table de contingence (comptage des saisons par cluster)
contingency_table = pd.crosstab(data_for_test['Cluster'], data_for_test['Saison'])

# Afficher la table de contingence
print("Table de contingence des saisons par cluster :")
print(contingency_table)

# Appliquer le test de Chi-Carré
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Afficher les résultats du test
print(f"\nRésultats du test de Chi-Carré:")
print(f"Statistique Chi-Carré = {chi2}")
print(f"Degrés de liberté = {dof}")
print(f"p-value = {p}")

# Interprétation des résultats
alpha = 0.05  # Niveau de signification

if p < alpha:
    print("Nous rejetons H0. La distribution des saisons est significativement différente entre les clusters.")
else:
    print("Nous ne rejetons pas H0. Il n'y a pas de différence significative dans la répartition des saisons entre les clusters.")

---
###### 3.5.1.2 TSNE

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=3, random_state=42, perplexity=100)
X_tsne = tsne.fit_transform(X_scaled)

# 📊 Initialisation des listes
wcss = []
sil_scores = []
db_scores = []

# 🔁 Boucle sur différents k
for k in range(1, 26):
    kmeans = KMeans(n_clusters=k, init='k-means++', max_iter=300, n_init=10, random_state=42)
    kmeans.fit(X_tsne)

    wcss.append(kmeans.inertia_)

    if k > 1:
        sil_scores.append(silhouette_score(X_tsne, kmeans.labels_))
        db_scores.append(davies_bouldin_score(X_tsne, kmeans.labels_))

# 📈 Silhouette Score
plt.figure(figsize=(10, 6))
plt.plot(range(2, 26), sil_scores, marker='o', linestyle='--', color='green')
plt.title('📊 Silhouette Score par nombre de clusters')
plt.xlabel('Nombre de clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

# 📈 Davies-Bouldin Index
plt.figure(figsize=(10, 6))
plt.plot(range(2, 26), db_scores, marker='o', linestyle='--', color='orange')
plt.title('📉 Davies-Bouldin Index par nombre de clusters')
plt.xlabel('Nombre de clusters')
plt.ylabel("Davies-Bouldin Index (↓ mieux)")
plt.grid(True)
plt.show()

# 🔍 Nombre optimal basé sur Silhouette
optimal_k = sil_scores.index(max(sil_scores)) + 2
print(f"✅ Nombre optimal de clusters selon le Silhouette Score : {optimal_k}")

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import numpy as np

# 🤖 Appliquer KMeans sur les données de t-SNE (en 3D)
kmeans = KMeans(n_clusters=16, random_state=42)
kmeans_TSNE_labels = kmeans.fit_predict(X_tsne)
centroids = kmeans.cluster_centers_

# 👨‍💻 Appliquer t-SNE sur les 3 premières composantes principales
tsne = TSNE(n_components=3, random_state=42, perplexity=100)
X_tsne_3d = tsne.fit_transform(X_scaled[:, :3])  # Utilisation des 3 premières composantes PCA

# 📊 Visualisation avec différentes combinaisons de composantes
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# Composantes 1 avec 2
axes[0].scatter(X_tsne_3d[:, 0], X_tsne_3d[:, 1], c=kmeans_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[0].set_title('Composante 1 vs Composante 2')
axes[0].set_xlabel('Composante 1')
axes[0].set_ylabel('Composante 2')
axes[0].grid(True)

# Composantes 2 avec 3
axes[1].scatter(X_tsne_3d[:, 1], X_tsne_3d[:, 2], c=kmeans_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[1].set_title('Composante 2 vs Composante 3')
axes[1].set_xlabel('Composante 2')
axes[1].set_ylabel('Composante 3')
axes[1].grid(True)

# Composantes 1 avec 3
axes[2].scatter(X_tsne_3d[:, 0], X_tsne_3d[:, 2], c=kmeans_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[2].set_title('Composante 1 vs Composante 3')
axes[2].set_xlabel('Composante 1')
axes[2].set_ylabel('Composante 3')
axes[2].grid(True)

# Affichage des centres des clusters
for ax in axes:
    ax.scatter(centroids[:, 0], centroids[:, 1], c='red', s=150, marker='X', label='Centres des clusters')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import folium
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.cm as cm

# Assure-toi que `kmeans_PCA_labels` est déjà défini
# Ici, kmeans_PCA_labels = kmeans.labels_

# Créer une carte centrée sur la France
map_france = folium.Map(location=[46.603354, 1.888334], zoom_start=6)  # Coordonnées géographiques approximatives de la France

# Générer 16 couleurs distinctes en utilisant matplotlib
colors = cm.get_cmap('tab20', 16)  # 'tab20' génère 20 couleurs distinctes
cluster_colors = [colors(i) for i in range(16)]  # Attribuer une couleur à chaque cluster

# Afficher les points des clusters sur la carte
for i, row in data_filtered.iterrows():
    # Limiter les valeurs des coordonnées pour s'assurer qu'elles sont dans les bornes de la France
    latitude = np.clip(row['latitude'], 42, 51)
    longitude = np.clip(row['longitude'], -5, 10)

    # Utiliser kmeans_PCA_labels pour récupérer le numéro du cluster pour chaque point
    cluster_label = kmeans_TSNE_labels[i]

    # Attribuer une couleur en fonction du cluster
    cluster_color = cluster_colors[cluster_label]

    # Créer un popup avec le numéro du cluster
    popup = folium.Popup(f'Cluster: {cluster_label}', parse_html=True)

    folium.CircleMarker(
        location=[latitude, longitude],
        radius=5,
        color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill=True,
        fill_color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill_opacity=0.6,
        popup=popup  # Ajouter le popup ici
    ).add_to(map_france)

# Afficher la carte dans un fichier HTML
map_france.save('../results_img_csv/map_clusters_kmeans_TSNE.html')

# Afficher un message de succès
print("✅ Carte avec les points des clusters et leurs numéros enregistrée sous 'map_clusters_kmeans_TSNE.html'.")

In [ ]:
from scipy.spatial import Voronoi, voronoi_plot_2d

# Dictionnaire des couleurs des saisons
saison_colors = {
    'hiver': '#1f77b4',
    'printemps': '#2ca02c',
    'été': '#ff7f0e',
    'automne': '#d62728'
}

# 🤖 Clustering KMeans
kmeans = KMeans(n_clusters=16, random_state=42)
kmeans_TSNE_labels = kmeans.fit_predict(X_tsne)
centroids = kmeans.cluster_centers_

# Récupérer les saisons en minuscules
saisons = data_filtered["Saison"].str.lower().values

# 📊 Visualisation avec t-SNE
plt.figure(figsize=(10, 8))

for cluster_id in range(16):
    cluster_mask = (kmeans_TSNE_labels == cluster_id)
    cluster_points = X_tsne[cluster_mask]
    cluster_saisons = saisons[cluster_mask]

    # Points colorés par saison dans le cluster
    for saison in np.unique(cluster_saisons):
        saison_mask = (cluster_saisons == saison)
        plt.scatter(
            cluster_points[saison_mask, 0],
            cluster_points[saison_mask, 1],
            color=saison_colors[saison],
            label=f'Cluster {cluster_id} - {saison.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # 🔢 Affichage du numéro du cluster au centre
    plt.text(
        centroids[cluster_id, 0], centroids[cluster_id, 1],
        str(cluster_id), fontsize=14, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# Finalisation du plot
plt.title(f'Clustering KMeans avec t-SNE + délimitation, coloré par saison')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.spatial import Voronoi, voronoi_plot_2d
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
import unidecode

# 🎨 Couleurs des natures
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 🤖 Clustering KMeans sur les données t-SNE
kmeans = KMeans(n_clusters=16, random_state=42)
kmeans_TSNE_labels = kmeans.fit_predict(X_tsne)
centroids = kmeans.cluster_centers_

# 🔤 Nettoyage et normalisation des natures
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

natures_raw = data_filtered["Nature"].str.strip().str.lower()
natures = natures_raw.apply(normalize_nature)
natures = natures.replace({
    'accidentel': 'accidentelle',
    'involontaire(particulier)': 'involontaire (particulier)',
    'involontaire ( particulier )': 'involontaire (particulier)',
    'involontaire(travaux)': 'involontaire (travaux)',
    'malveillance': 'malveillance',
    'naturelle': 'naturelle'
})

# 📊 Visualisation
plt.figure(figsize=(10, 8))

for cluster_id in range(14):
    cluster_mask = (kmeans_TSNE_labels == cluster_id)
    cluster_points = X_tsne[cluster_mask]
    cluster_natures = natures[cluster_mask]

    # Points colorés par nature dans le cluster
    for nature in np.unique(cluster_natures):
        nature_mask = (cluster_natures == nature)
        plt.scatter(
            cluster_points[nature_mask, 0],
            cluster_points[nature_mask, 1],
            color=nature_colors.get(nature, '#999999'),
            edgecolors='k',
            s=60,
            alpha=0.7
        )

# 🔢 Affichage des numéros des clusters
for cluster_id in range(16):
    plt.text(
        centroids[cluster_id, 0], centroids[cluster_id, 1],
        str(cluster_id), fontsize=14, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# ➕ Légende simplifiée (1 fois par nature)
handles = [
    plt.Line2D([0], [0], marker='o', color='w', label=nature.capitalize(),
               markerfacecolor=color, markeredgecolor='k', markersize=10)
    for nature, color in nature_colors.items()
]
plt.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left')

# Finalisation du plot
plt.title(f'Clustering KMeans avec t-SNE + délimitation, coloré par nature')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Créer un DataFrame avec les labels de KMeans et les saisons
data_for_test = pd.DataFrame({
    'Cluster': kmeans_TSNE_labels,  # Les labels des clusters KMeans
    'Saison': data_filtered['Saison'].str.lower()  # Les saisons (en minuscule)
})

# Créer une table de contingence (comptage des saisons par cluster)
contingency_table = pd.crosstab(data_for_test['Cluster'], data_for_test['Saison'])

# Afficher la table de contingence
print("Table de contingence des saisons par cluster :")
print(contingency_table)

# Appliquer le test de Chi-Carré
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Afficher les résultats du test
print(f"\nRésultats du test de Chi-Carré:")
print(f"Statistique Chi-Carré = {chi2}")
print(f"Degrés de liberté = {dof}")
print(f"p-value = {p}")

# Interprétation des résultats
alpha = 0.05  # Niveau de signification

if p < alpha:
    print("Nous rejetons H0. La distribution des saisons est significativement différente entre les clusters.")
else:
    print("Nous ne rejetons pas H0. Il n'y a pas de différence significative dans la répartition des saisons entre les clusters.")

---
###### 3.5.1.3 PCA combiné à TSNE

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

# Appliquer PCA pour réduire la dimensionnalité
pca = PCA(n_components=20)  # On peut ajuster ce nombre selon les besoins
X_pca = pca.fit_transform(X_scaled)  # X : données d'origine

# Appliquer t-SNE sur les résultats de PCA
tsne = TSNE(n_components=3, random_state=42)  # On réduit à 2D pour la visualisation
X_tsne = tsne.fit_transform(X_pca)  # Réduction supplémentaire des dimensions

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Liste pour stocker les scores de silhouette
silhouette_scores = []
range_n_clusters = range(2, 31)  # Tester de 2 à 10 clusters (ajustable)

# Appliquer KMeans pour chaque nombre de clusters dans la plage spécifiée
for n_clusters in range_n_clusters:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans_labels = kmeans.fit_predict(X_tsne)  # Appliquer KMeans sur les données réduites
    score = silhouette_score(X_tsne, kmeans_labels)  # Calculer le silhouette score
    silhouette_scores.append(score)

# Tracer le silhouette score pour chaque nombre de clusters
plt.figure(figsize=(8, 6))
plt.plot(range_n_clusters, silhouette_scores, marker='o', color='b', linestyle='-', markersize=8)
plt.title('Silhouette Score pour déterminer le nombre optimal de clusters')
plt.xlabel('Nombre de clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

# Afficher le nombre optimal de clusters basé sur le silhouette score
optimal_n_clusters = range_n_clusters[np.argmax(silhouette_scores)]
print(f"Le nombre optimal de clusters basé sur le silhouette score est : {optimal_n_clusters}")

In [ ]:
# Appliquer KMeans avec le nombre optimal de clusters trouvé (par exemple, 4 clusters)
kmeans = KMeans(n_clusters=15, random_state=42)
kmeans_labels = kmeans.fit_predict(X_tsne)  # Appliquer KMeans sur les données réduites

# Visualisation des résultats avec les clusters détectés
plt.figure(figsize=(10, 8))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=kmeans_labels, cmap='viridis', s=50, alpha=0.6, edgecolors='k')

# Ajouter les centres des clusters (KMeans centroids)
centroids = kmeans.cluster_centers_
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, marker='X', label='Centres des clusters')

# Final touches
plt.xlabel('Composante 1')
plt.ylabel('Composante 2')
plt.grid(True)
plt.legend()
plt.show()

---
##### 3.5.2 GMM

---
###### 3.5.2.1 PCA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# 🔧 Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 📉 Réduction dimensionnelle avec PCA
pca = PCA(n_components=20)
X_pca = pca.fit_transform(X_scaled)

# 📊 Initialisation des listes
sil_scores = []
db_scores = []
aic_scores = []
bic_scores = []

# 🔁 Boucle sur différents k
for k in range(1, 31):
    gmm = GaussianMixture(n_components=k, random_state=42)
    gmm.fit(X_pca)

    # ✅ AIC & BIC
    aic_scores.append(gmm.aic(X_pca))
    bic_scores.append(gmm.bic(X_pca))

    if k > 1:
        labels = gmm.predict(X_pca)
        sil_scores.append(silhouette_score(X_pca, labels))
        db_scores.append(davies_bouldin_score(X_pca, labels))

# 📈 AIC & BIC
plt.figure(figsize=(10, 6))
plt.plot(range(1, 31), aic_scores, marker='o', linestyle='--', label='AIC')
plt.plot(range(1, 31), bic_scores, marker='s', linestyle='--', label='BIC')
plt.title("AIC & BIC pour GMM avec PCA")
plt.xlabel("Nombre de clusters")
plt.ylabel("Score (↓ mieux)")
plt.legend()
plt.grid(True)
plt.show()

# 📈 Silhouette Score
plt.figure(figsize=(10, 6))
plt.plot(range(2, 31), sil_scores, marker='o', color='g', linestyle='--')
plt.title("Silhouette Score pour GMM avec PCA")
plt.xlabel("Nombre de clusters")
plt.ylabel("Silhouette Score")
plt.grid(True)
plt.show()

# 📈 Davies-Bouldin Index
plt.figure(figsize=(10, 6))
plt.plot(range(2, 31), db_scores, marker='o', linestyle='--', color='orange')
plt.title("Davies-Bouldin Index pour GMM avec PCA")
plt.xlabel("Nombre de clusters")
plt.ylabel("Davies-Bouldin Index (↓ mieux)")
plt.grid(True)
plt.show()

# ✅ Nombre optimal selon Silhouette
optimal_k_sil = sil_scores.index(max(sil_scores)) + 2
print(f"✅ Nombre optimal selon le Silhouette Score : {optimal_k_sil}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.mixture import GaussianMixture
from scipy.spatial import ConvexHull

# Appliquer GMM
gmm = GaussianMixture(n_components=13, random_state=42)
gmm.fit(X_pca)

# Récupérer les labels et les centres
gmm_PCA_labels = gmm.predict(X_pca)
centroids = gmm.means_
# Visualisation
plt.figure(figsize=(10, 8))
plt.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=gmm_PCA_labels, cmap='plasma',
    s=50, marker='o', edgecolors='k', alpha=0.6
)
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, marker='X', label='Centres des clusters')

# Final touches
plt.title(f'Clustering GMM avec {optimal_k} clusters (PCA réduit à 2 dimensions)')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import folium
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.cm as cm

# Assure-toi que `kmeans_PCA_labels` est déjà défini
# Ici, kmeans_PCA_labels = kmeans.labels_

# Créer une carte centrée sur la France
map_france = folium.Map(location=[46.603354, 1.888334], zoom_start=6)  # Coordonnées géographiques approximatives de la France

# Générer 16 couleurs distinctes en utilisant matplotlib
colors = cm.get_cmap('tab20', 13)  # 'tab20' génère 20 couleurs distinctes
cluster_colors = [colors(i) for i in range(13)]  # Attribuer une couleur à chaque cluster

# Afficher les points des clusters sur la carte
for i, row in data_filtered.iterrows():
    # Limiter les valeurs des coordonnées pour s'assurer qu'elles sont dans les bornes de la France
    latitude = np.clip(row['latitude'], 42, 51)
    longitude = np.clip(row['longitude'], -5, 10)

    # Utiliser kmeans_PCA_labels pour récupérer le numéro du cluster pour chaque point
    cluster_label = gmm_PCA_labels[i]

    # Attribuer une couleur en fonction du cluster
    cluster_color = cluster_colors[cluster_label]

    # Créer un popup avec le numéro du cluster
    popup = folium.Popup(f'Cluster: {cluster_label}', parse_html=True)

    folium.CircleMarker(
        location=[latitude, longitude],
        radius=5,
        color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill=True,
        fill_color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill_opacity=0.6,
        popup=popup  # Ajouter le popup ici
    ).add_to(map_france)

# Afficher la carte dans un fichier HTML
map_france.save('../results_img_csv/map_clusters_gmm_PCA.html')

# Afficher un message de succès
print("✅ Carte avec les points des clusters et leurs numéros enregistrée sous 'map_clusters_gmm_PCA.html'.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.mixture import GaussianMixture
from scipy.spatial import ConvexHull

# Dictionnaire des couleurs des saisons
saison_colors = {
    'hiver': '#1f77b4',
    'printemps': '#2ca02c',
    'été': '#ff7f0e',
    'automne': '#d62728'
}

# Appliquer GMM
gmm = GaussianMixture(n_components=13, random_state=42)
gmm.fit(X_pca)

# Récupérer les labels et les centres
gmm_PCA_labels = gmm.predict(X_pca)
centroids = gmm.means_

X_pca_visu = X_pca[:, :2]
centroids_2D = centroids[:, :2]

# Récupérer les saisons en minuscules
saisons = data_filtered["Saison"].str.lower().values

# Visualisation
plt.figure(figsize=(10, 8))

for cluster_id in range(13):
    cluster_mask = (gmm_PCA_labels == cluster_id)
    cluster_points = X_pca_visu[cluster_mask]
    cluster_saisons = saisons[cluster_mask]

    # Points colorés par saison dans le cluster
    for saison in np.unique(cluster_saisons):
        saison_mask = (cluster_saisons == saison)
        plt.scatter(
            cluster_points[saison_mask, 0],
            cluster_points[saison_mask, 1],
            color=saison_colors[saison],
            label=f'Cluster {cluster_id} - {saison.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # 🔢 Affichage du numéro du cluster au centre
    plt.text(
        centroids_2D[cluster_id, 0], centroids_2D[cluster_id, 1],
        str(cluster_id), fontsize=14, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# Finalisation du plot
plt.title(f'Clustering GMM (PCA) avec {7} clusters, colorés par saison')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.mixture import GaussianMixture
from scipy.spatial import ConvexHull

# 🎨 Dictionnaire des couleurs pour chaque nature
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 🎯 Appliquer GMM
gmm = GaussianMixture(n_components=13, random_state=42)
gmm.fit(X_pca)

# 🎯 Récupération des clusters et centres
gmm_PCA_labels = gmm.predict(X_pca)
centroids = gmm.means_

X_pca_visu = X_pca[:, :2]
centroids_2D = centroids[:, :2]
import unidecode

# Fonction de normalisation des chaînes
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)  # retire les accents
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

# Appliquer la normalisation
natures_raw = data_filtered["Nature"].str.strip().str.lower()
natures = natures_raw.apply(normalize_nature)

# Mapping vers noms propres cohérents
natures = natures.replace({
    'accidentel': 'accidentelle',
    'involontaire ( particulier )': 'involontaire (particulier)',
    'involontaire(particulier)': 'involontaire (particulier)',
    'involontaire(travaux)': 'involontaire (travaux)',
    'naturelle': 'naturelle',
    'malveillance': 'malveillance',
})

# → vérif rapide
print(natures.unique())

# 📊 Visualisation
plt.figure(figsize=(10, 8))

# ➕ Tracer tous les clusters
for cluster_id in range(13):
    cluster_mask = (gmm_PCA_labels == cluster_id)
    cluster_points = X_pca_visu[cluster_mask]
    cluster_natures = natures[cluster_mask]

    # ➕ Points colorés par nature dans le cluster
    for nature in np.unique(cluster_natures):
        nature_mask = (cluster_natures == nature)
        plt.scatter(
            cluster_points[nature_mask, 0],
            cluster_points[nature_mask, 1],
            color=nature_colors.get(nature, '#999999'),
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # 🔢 Affichage du numéro du cluster au centre
    plt.text(
        centroids_2D[cluster_id, 0], centroids_2D[cluster_id, 1],
        str(cluster_id), fontsize=14, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# ➕ Légende simplifiée : une fois par nature
handles = [
    plt.Line2D([0], [0], marker='o', color='w', label=nature.capitalize(),
               markerfacecolor=color, markeredgecolor='k', markersize=10)
    for nature, color in nature_colors.items()
]
plt.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left')

# Finalisation du plot
plt.title(f'Clustering GMM (PCA) avec {7} clusters, colorés par nature')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Créer un DataFrame avec les labels de KMeans et les saisons
data_for_test = pd.DataFrame({
    'Cluster': gmm_PCA_labels,  # Les labels des clusters KMeans
    'Saison': data_filtered['Saison'].str.lower()  # Les saisons (en minuscule)
})

# Créer une table de contingence (comptage des saisons par cluster)
contingency_table = pd.crosstab(data_for_test['Cluster'], data_for_test['Saison'])

# Afficher la table de contingence
print("Table de contingence des saisons par cluster :")
print(contingency_table)

# Appliquer le test de Chi-Carré
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Afficher les résultats du test
print(f"\nRésultats du test de Chi-Carré:")
print(f"Statistique Chi-Carré = {chi2}")
print(f"Degrés de liberté = {dof}")
print(f"p-value = {p}")

# Interprétation des résultats
alpha = 0.05  # Niveau de signification

if p < alpha:
    print("Nous rejetons H0. La distribution des saisons est significativement différente entre les clusters.")
else:
    print("Nous ne rejetons pas H0. Il n'y a pas de différence significative dans la répartition des saisons entre les clusters.")

---
###### 3.5.2.2 TSNE

In [ ]:
tsne = TSNE(n_components=3, random_state=42, perplexity=100)
X_tsne = tsne.fit_transform(X_scaled)

# Calculer le Silhouette Score pour différents nombres de clusters
sil_scores = []
db_scores = []
for k in range(2, 31):  # Teste de 2 à 10 clusters
    gmm = GaussianMixture(n_components=k, random_state=42)
    gmm_labels = gmm.fit_predict(X_tsne)  # Appliquer Agglomerative Clustering
    sil_scores.append(silhouette_score(X_tsne, gmm_labels))  # Calcul du Silhouette Score
    db_scores.append(davies_bouldin_score(X_tsne, gmm_labels))

# Tracer le Silhouette Score
plt.figure(figsize=(10, 6))
plt.plot(range(2, 31), sil_scores, marker='o', color='g', linestyle='--')
plt.title('Silhouette Score pour Agglomerative Clustering avec PCA')
plt.xlabel('Nombre de clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

# 📈 Davies-Bouldin Index
plt.figure(figsize=(10, 6))
plt.plot(range(2, 31), db_scores, marker='o', linestyle='--', color='orange')
plt.title('📉 Davies-Bouldin Index par nombre de clusters')
plt.xlabel('Nombre de clusters')
plt.ylabel("Davies-Bouldin Index (↓ mieux)")
plt.grid(True)
plt.show()

# Nombre optimal de clusters basé sur le Silhouette Score
optimal_k = sil_scores.index(max(sil_scores)) + 2  # Les indices de sil_scores commencent à 2
print(f"Nombre optimal de clusters selon le Silhouette Score : {optimal_k}")

In [ ]:
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial import ConvexHull

# 🤖 Clustering avec GMM
gmm = GaussianMixture(n_components=12, random_state=42)
gmm_TSNE_labels = gmm.fit_predict(X_tsne)

# 👨‍💻 Visualisation avec différentes combinaisons de composantes t-SNE
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# Composantes 1 avec 2
axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=gmm_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[0].set_title('Composante 1 vs Composante 2')
axes[0].set_xlabel('Composante 1')
axes[0].set_ylabel('Composante 2')
axes[0].grid(True)

# Composantes 2 avec 3
axes[1].scatter(X_tsne[:, 1], X_tsne[:, 2], c=gmm_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[1].set_title('Composante 2 vs Composante 3')
axes[1].set_xlabel('Composante 2')
axes[1].set_ylabel('Composante 3')
axes[1].grid(True)

# Composantes 1 avec 3
axes[2].scatter(X_tsne[:, 0], X_tsne[:, 2], c=gmm_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[2].set_title('Composante 1 vs Composante 3')
axes[2].set_xlabel('Composante 1')
axes[2].set_ylabel('Composante 3')
axes[2].grid(True)

# Affichage final
plt.tight_layout()
plt.show()

In [ ]:
import folium
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.cm as cm

# Assure-toi que `kmeans_PCA_labels` est déjà défini
# Ici, kmeans_PCA_labels = kmeans.labels_

# Créer une carte centrée sur la France
map_france = folium.Map(location=[46.603354, 1.888334], zoom_start=6)  # Coordonnées géographiques approximatives de la France

# Générer 16 couleurs distinctes en utilisant matplotlib
colors = cm.get_cmap('tab20', 12)  # 'tab20' génère 20 couleurs distinctes
cluster_colors = [colors(i) for i in range(12)]  # Attribuer une couleur à chaque cluster

# Afficher les points des clusters sur la carte
for i, row in data_filtered.iterrows():
    # Limiter les valeurs des coordonnées pour s'assurer qu'elles sont dans les bornes de la France
    latitude = np.clip(row['latitude'], 42, 51)
    longitude = np.clip(row['longitude'], -5, 10)

    # Utiliser kmeans_PCA_labels pour récupérer le numéro du cluster pour chaque point
    cluster_label = gmm_TSNE_labels[i]

    # Attribuer une couleur en fonction du cluster
    cluster_color = cluster_colors[cluster_label]

    # Créer un popup avec le numéro du cluster
    popup = folium.Popup(f'Cluster: {cluster_label}', parse_html=True)

    folium.CircleMarker(
        location=[latitude, longitude],
        radius=5,
        color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill=True,
        fill_color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill_opacity=0.6,
        popup=popup  # Ajouter le popup ici
    ).add_to(map_france)

# Afficher la carte dans un fichier HTML
map_france.save('../results_img_csv/map_clusters_gmm_TSNE.html')

# Afficher un message de succès
print("✅ Carte avec les points des clusters et leurs numéros enregistrée sous 'map_clusters_gmm_TSNE.html'.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.mixture import GaussianMixture

# Dictionnaire des couleurs des saisons
saison_colors = {
    'hiver': '#1f77b4',
    'printemps': '#2ca02c',
    'été': '#ff7f0e',
    'automne': '#d62728'
}

# 🤖 Clustering avec GMM
gmm = GaussianMixture(n_components=12, random_state=42)
gmm_TSNE_labels = gmm.fit_predict(X_tsne)
centroids = gmm.means_  # Centres dans l'espace t-SNE

# Récupérer les saisons en minuscules
saisons = data_filtered["Saison"].str.lower().values

# 📊 Visualisation avec t-SNE
plt.figure(figsize=(10, 8))

for cluster_id in np.unique(gmm_TSNE_labels):
    cluster_mask = (gmm_TSNE_labels == cluster_id)
    cluster_points = X_tsne[cluster_mask]
    cluster_saisons = saisons[cluster_mask]

    # Points colorés par saison dans le cluster
    for saison in np.unique(cluster_saisons):
        saison_mask = (cluster_saisons == saison)
        plt.scatter(
            cluster_points[saison_mask, 0],
            cluster_points[saison_mask, 1],
            color=saison_colors[saison],
            label=f'Cluster {cluster_id} - {saison.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # 🔢 Ajouter le numéro du cluster au centre
    plt.text(
        centroids[cluster_id, 0], centroids[cluster_id, 1],
        str(cluster_id), fontsize=14, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# 🏷️ Titres et labels
plt.title('Visualisation des clusters GMM avec t-SNE (colorés par saison)')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.mixture import GaussianMixture
import unidecode

# 🎨 Dictionnaire des couleurs pour chaque nature
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillante': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 🔤 Normalisation des natures
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

natures = data_filtered["Nature"].str.strip().str.lower().apply(normalize_nature)
natures = natures.replace({
    'accidentel': 'accidentelle',
    'involontaire ( particulier )': 'involontaire (particulier)',
    'involontaire(particulier)': 'involontaire (particulier)',
    'involontaire(travaux)': 'involontaire (travaux)',
    'malveillance': 'malveillante',
})

# 📊 Visualisation t-SNE
plt.figure(figsize=(10, 8))

# 💡 Ajouter les centres de clusters
gmm = GaussianMixture(n_components=12, random_state=42)
gmm.fit(X_tsne)
gmm_TSNE_labels = gmm.predict(X_tsne)
centroids_tsne = gmm.means_

for cluster_id in range(12):
    cluster_mask = (gmm_TSNE_labels == cluster_id)
    cluster_points = X_tsne[cluster_mask]
    cluster_natures = natures[cluster_mask]

    for nature in np.unique(cluster_natures):
        nature_mask = (cluster_natures == nature)
        plt.scatter(
            cluster_points[nature_mask, 0],
            cluster_points[nature_mask, 1],
            color=nature_colors.get(nature, '#999999'),
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # 🔢 Ajouter numéro du cluster
    plt.text(
        centroids_tsne[cluster_id, 0], centroids_tsne[cluster_id, 1],
        str(cluster_id), fontsize=14, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# ➕ Légende simplifiée
handles = [
    plt.Line2D([0], [0], marker='o', color='w', label=nature.capitalize(),
               markerfacecolor=color, markeredgecolor='k', markersize=10)
    for nature, color in nature_colors.items()
]
plt.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left')

# Finalisation
plt.title(f'Clustering GMM (t-SNE) avec 12 clusters, colorés par nature')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Créer un DataFrame avec les labels de KMeans et les saisons
data_for_test = pd.DataFrame({
    'Cluster': gmm_TSNE_labels,  # Les labels des clusters KMeans
    'Saison': data_filtered['Saison'].str.lower()  # Les saisons (en minuscule)
})

# Créer une table de contingence (comptage des saisons par cluster)
contingency_table = pd.crosstab(data_for_test['Cluster'], data_for_test['Saison'])

# Afficher la table de contingence
print("Table de contingence des saisons par cluster :")
print(contingency_table)

# Appliquer le test de Chi-Carré
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Afficher les résultats du test
print(f"\nRésultats du test de Chi-Carré:")
print(f"Statistique Chi-Carré = {chi2}")
print(f"Degrés de liberté = {dof}")
print(f"p-value = {p}")

# Interprétation des résultats
alpha = 0.05  # Niveau de signification

if p < alpha:
    print("Nous rejetons H0. La distribution des saisons est significativement différente entre les clusters.")
else:
    print("Nous ne rejetons pas H0. Il n'y a pas de différence significative dans la répartition des saisons entre les clusters.")

---
###### 3.5.2.3 PCA combiné avec TSNE

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

# Appliquer PCA pour réduire la dimensionnalité
pca = PCA(n_components=20)  # On peut ajuster ce nombre selon les besoins
X_pca = pca.fit_transform(X_scaled)  # X : données d'origine

# Appliquer t-SNE sur les résultats de PCA
tsne = TSNE(n_components=3, random_state=42, perplexity=100)  # On réduit à 2D pour la visualisation
X_tsne = tsne.fit_transform(X_pca)  # Réduction supplémentaire des dimensions

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Liste pour stocker les scores de silhouette
silhouette_scores = []
range_n_clusters = range(2, 31)  # Tester de 2 à 10 clusters (ajustable)

# Appliquer KMeans pour chaque nombre de clusters dans la plage spécifiée
for n_clusters in range_n_clusters:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans_labels = kmeans.fit_predict(X_tsne)  # Appliquer KMeans sur les données réduites
    score = silhouette_score(X_tsne, kmeans_labels)  # Calculer le silhouette score
    silhouette_scores.append(score)

# Tracer le silhouette score pour chaque nombre de clusters
plt.figure(figsize=(8, 6))
plt.plot(range_n_clusters, silhouette_scores, marker='o', color='b', linestyle='-', markersize=8)
plt.title('Silhouette Score pour déterminer le nombre optimal de clusters')
plt.xlabel('Nombre de clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

# Afficher le nombre optimal de clusters basé sur le silhouette score
optimal_n_clusters = range_n_clusters[np.argmax(silhouette_scores)]
print(f"Le nombre optimal de clusters basé sur le silhouette score est : {optimal_n_clusters}")

In [ ]:
# Appliquer KMeans avec le nombre optimal de clusters trouvé
kmeans = KMeans(n_clusters=15, random_state=42)
kmeans_labels = kmeans.fit_predict(X_tsne)  # Appliquer KMeans sur les données réduites

# Visualisation des résultats avec les clusters détectés
plt.figure(figsize=(10, 8))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=kmeans_labels, cmap='viridis', s=50, alpha=0.6, edgecolors='k')

# Ajouter les centres des clusters (KMeans centroids)
centroids = kmeans.cluster_centers_
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, marker='X', label='Centres des clusters')

# Final touches
plt.xlabel('Composante 1')
plt.ylabel('Composante 2')
plt.grid(True)
plt.legend()
plt.show()

---
##### 3.5.3 DBSCAN

---
###### 3.5.3.1 PCA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import MinMaxScaler

# 📌 Normalisation
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# 📉 Réduction dimensionnelle avec PCA (2D pour visualisation si besoin)
pca = PCA(n_components=20)
X_pca = pca.fit_transform(X_scaled)

# 🔍 Paramètres à tester
eps_values = np.arange(0.1, 5.0, 0.2)
min_samples = 5  # Tu peux faire varier aussi si tu veux

sil_scores = []
db_scores = []
n_clusters_list = []

for eps in eps_values:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_pca)

    # Nombre de clusters (hors bruit)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_clusters_list.append(n_clusters)

    # Filtrer le bruit pour calcul des métriques
    if n_clusters > 1:
        mask = labels != -1
        sil = silhouette_score(X_pca[mask], labels[mask])
        db = davies_bouldin_score(X_pca[mask], labels[mask])
        sil_scores.append(sil)
        db_scores.append(db)
    else:
        sil_scores.append(np.nan)
        db_scores.append(np.nan)

# 📈 Silhouette Score
plt.figure(figsize=(10, 6))
plt.plot(eps_values, sil_scores, marker='o', linestyle='--', color='green')
plt.title("📊 Silhouette Score selon eps (DBSCAN)")
plt.xlabel("Valeur de eps")
plt.ylabel("Silhouette Score")
plt.grid(True)
plt.show()

# 📈 Davies-Bouldin Index
plt.figure(figsize=(10, 6))
plt.plot(eps_values, db_scores, marker='o', linestyle='--', color='orange')
plt.title("📉 Davies-Bouldin Index selon eps (DBSCAN)")
plt.xlabel("Valeur de eps")
plt.ylabel("Davies-Bouldin Index (↓ mieux)")
plt.grid(True)
plt.show()

# 🔍 Meilleur eps selon silhouette
best_index = np.nanargmax(sil_scores)
best_eps = eps_values[best_index]
print(f"✅ Valeur optimale de eps selon le Silhouette Score : {best_eps:.2f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import DBSCAN
from scipy.spatial import ConvexHull

# 1. 📌 Normalisation des données
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)  # Remplace X par tes données brutes

# 2. 📉 Réduction de dimension avec PCA
pca = PCA(n_components=20)
X_pca = pca.fit_transform(X_scaled)

# 3. 📍 Utilisation du meilleur eps trouvé précédemment
best_eps = 0.3   # ← change cette valeur selon ce que tu as trouvé plus tôt
best_min_samples = 10  # ← idem, change si tu avais testé autre chose
X_pca_visu = X_pca[:, :2]
centroids_2D = centroids[:, :2]
# 4. 🔄 Appliquer DBSCAN avec ces hyperparamètres
dbscan_best = DBSCAN(eps=best_eps, min_samples=best_min_samples)
dbscan_PCA_labels_best = dbscan_best.fit_predict(X_pca_visu)

# 5. 🎨 Affichage des clusters
plt.figure(figsize=(10, 8))
unique_labels = set(dbscan_PCA_labels_best)
colors = plt.cm.plasma(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, colors):
    if label == -1:
        color = 'black'
        label_name = "Bruit"
    else:
        label_name = f"Cluster {label}"

    # Points du cluster
    cluster_points = X_pca_visu[dbscan_PCA_labels_best == label]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1], color=color, label=label_name, edgecolors='k', alpha=0.6, s=60)

    # Tracer les contours pour les clusters (convex hull)
    if label != -1 and len(cluster_points) >= 3:
        hull = ConvexHull(cluster_points)
        for simplex in hull.simplices:
            plt.plot(cluster_points[simplex, 0], cluster_points[simplex, 1], 'k--', linewidth=1.2)

plt.title(f'🔍 Clustering DBSCAN avec eps={best_eps}, min_samples={best_min_samples}')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import DBSCAN
from scipy.spatial import ConvexHull

# Dictionnaire des couleurs des saisons
saison_colors = {
    'hiver': '#1f77b4',
    'printemps': '#2ca02c',
    'été': '#ff7f0e',
    'automne': '#d62728'
}

# 1. 📌 Normalisation des données
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)  # Remplace X par tes données brutes

# 2. 📉 Réduction de dimension avec PCA
pca = PCA(n_components=20)
X_pca = pca.fit_transform(X_scaled)

# 3. 📍 Utilisation du meilleur eps trouvé précédemment
best_eps = 0.5   # ← change cette valeur selon ce que tu as trouvé plus tôt
min_samples = 5  # ← idem, change si tu avais testé autre chose

X_pca_visu = X_pca[:, :2]
centroids_2D = centroids[:, :2]
# 4. 🔄 Appliquer DBSCAN avec ces hyperparamètres
dbscan_best = DBSCAN(eps=0.3, min_samples=10)
dbscan_PCA_labels_best = dbscan_best.fit_predict(X_pca_visu)

# Récupérer les saisons en minuscules
saisons = data_filtered["Saison"].str.lower().values

# 5. 🎨 Affichage des clusters
plt.figure(figsize=(10, 8))

# Pour chaque cluster
for label in np.unique(dbscan_PCA_labels_best):
    cluster_mask = (dbscan_PCA_labels_best == label)
    cluster_points = X_pca_visu[cluster_mask]
    cluster_saisons = saisons[cluster_mask]

    # Points colorés par saison dans le cluster
    for saison in np.unique(cluster_saisons):
        saison_mask = (cluster_saisons == saison)
        plt.scatter(
            cluster_points[saison_mask, 0],
            cluster_points[saison_mask, 1],
            color=saison_colors[saison],
            label=f'Cluster {label} - {saison.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # ➕ Lignes de délimitation du cluster (ConvexHull)
    if label != -1 and len(cluster_points) >= 3:
        hull = ConvexHull(cluster_points)
        for simplex in hull.simplices:
            plt.plot(cluster_points[simplex, 0], cluster_points[simplex, 1], 'k--', linewidth=1.2)

    # Si c'est du bruit, on marque un point noir
    if label == -1:
        plt.scatter(cluster_points[:, 0], cluster_points[:, 1], color='black', label='Bruit', edgecolors='k', s=60, alpha=0.7)

# Titres et légendes
plt.title(f'Clustering DBSCAN avec eps={0.3}, min_samples={10}, coloré par saison')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import DBSCAN
from scipy.spatial import ConvexHull
import unidecode

# 🎨 Dictionnaire des couleurs des natures
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 🔤 Fonction de normalisation
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

# ✔ Nettoyage des valeurs
natures_raw = data_filtered["Nature"].str.strip().str.lower()
natures = natures_raw.apply(normalize_nature)
natures = natures.replace({
    'accidentel': 'accidentelle',
    'involontaire(particulier)': 'involontaire (particulier)',
    'involontaire ( particulier )': 'involontaire (particulier)',
    'involontaire(travaux)': 'involontaire (travaux)',
    'malveillance': 'malveillance',
    'naturelle': 'naturelle'
})

# 1. 📌 Normalisation
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# 2. 📉 PCA
pca = PCA(n_components=20)
X_pca = pca.fit_transform(X_scaled)
X_pca_visu = X_pca[:, :2]
centroids_2D = centroids[:, :2]
# 3. 🔍 DBSCAN
best_eps = 0.3  # à ajuster si besoin
min_samples = 10
dbscan_best = DBSCAN(eps=0.3, min_samples=10)
dbscan_PCA_labels_best = dbscan_best.fit_predict(X_pca_visu)

# 4. 📊 Affichage
plt.figure(figsize=(10, 8))

for label in np.unique(dbscan_PCA_labels_best):
    cluster_mask = (dbscan_PCA_labels_best == label)
    cluster_points = X_pca_visu[cluster_mask]
    cluster_natures = natures[cluster_mask]

    # Points colorés par nature
    for nature in np.unique(cluster_natures):
        nature_mask = (cluster_natures == nature)
        plt.scatter(
            cluster_points[nature_mask, 0],
            cluster_points[nature_mask, 1],
            color=nature_colors.get(nature, '#999999'),
            label=f'Cluster {label} - {nature.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # ➕ ConvexHull si pas bruit
    if label != -1 and len(cluster_points) >= 3:
        hull = ConvexHull(cluster_points)
        for simplex in hull.simplices:
            plt.plot(cluster_points[simplex, 0], cluster_points[simplex, 1], 'k--', linewidth=1.2)

    # ➕ Bruit (label = -1)
    if label == -1:
        plt.scatter(cluster_points[:, 0], cluster_points[:, 1], color='black', label='Bruit', edgecolors='k', s=60, alpha=0.7)

# Légende simplifiée par nature
handles = [
    plt.Line2D([0], [0], marker='o', color='w', label=nature.capitalize(),
               markerfacecolor=color, markeredgecolor='k', markersize=10)
    for nature, color in nature_colors.items()
]
handles.append(plt.Line2D([0], [0], marker='o', color='w', label='Bruit',
                          markerfacecolor='black', markeredgecolor='k', markersize=10))

plt.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title(f'Clustering DBSCAN avec eps={best_eps}, min_samples={min_samples}, coloré par nature')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Créer un DataFrame avec les labels de KMeans et les saisons
data_for_test = pd.DataFrame({
    'Cluster': dbscan_PCA_labels_best,  # Les labels des clusters KMeans
    'Saison': data_filtered['Saison'].str.lower()  # Les saisons (en minuscule)
})

# Créer une table de contingence (comptage des saisons par cluster)
contingency_table = pd.crosstab(data_for_test['Cluster'], data_for_test['Saison'])

# Afficher la table de contingence
print("Table de contingence des saisons par cluster :")
print(contingency_table)

# Appliquer le test de Chi-Carré
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Afficher les résultats du test
print(f"\nRésultats du test de Chi-Carré:")
print(f"Statistique Chi-Carré = {chi2}")
print(f"Degrés de liberté = {dof}")
print(f"p-value = {p}")

# Interprétation des résultats
alpha = 0.05  # Niveau de signification

if p < alpha:
    print("Nous rejetons H0. La distribution des saisons est significativement différente entre les clusters.")
else:
    print("Nous ne rejetons pas H0. Il n'y a pas de différence significative dans la répartition des saisons entre les clusters.")

---
###### 3.5.3.2 TSNE

In [ ]:
tsne = TSNE(n_components=3, random_state=42, perplexity=100)
X_tsne = tsne.fit_transform(X_scaled)

# Calculer le Silhouette Score pour différents nombres de clusters
sil_scores = []
db_scores = []
for k in range(2, 31):  # Teste de 2 à 10 clusters
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    dbscan_labels = dbscan.fit_predict(X_tsne)  # Appliquer Agglomerative Clustering
    sil_scores.append(silhouette_score(X_tsne, dbscan_labels))  # Calcul du Silhouette Score
    db_scores.append(davies_bouldin_score(X_tsne, dbscan_labels))

# Tracer le Silhouette Score
plt.figure(figsize=(10, 6))
plt.plot(range(2, 31), sil_scores, marker='o', color='g', linestyle='--')
plt.title('Silhouette Score pour Agglomerative Clustering avec PCA')
plt.xlabel('Nombre de clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

# 📈 Davies-Bouldin Index
plt.figure(figsize=(10, 6))
plt.plot(range(2, 31), db_scores, marker='o', linestyle='--', color='orange')
plt.title('📉 Davies-Bouldin Index par nombre de clusters')
plt.xlabel('Nombre de clusters')
plt.ylabel("Davies-Bouldin Index (↓ mieux)")
plt.grid(True)
plt.show()

# Nombre optimal de clusters basé sur le Silhouette Score
optimal_k = sil_scores.index(max(sil_scores)) + 2  # Les indices de sil_scores commencent à 2
print(f"Nombre optimal de clusters selon le Silhouette Score : {optimal_k}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from scipy.spatial import ConvexHull

# Appliquer DBSCAN avec les paramètres spécifiés
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_TSNE_labels = dbscan.fit_predict(X_tsne)

# 📊 Visualisation avec différentes combinaisons de composantes t-SNE
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# 1ère combinaison : Composantes 1 avec 2
axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=dbscan_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[0].set_title('Composante 1 vs Composante 2')
axes[0].set_xlabel('Composante 1')
axes[0].set_ylabel('Composante 2')
axes[0].grid(True)

# 2ème combinaison : Composantes 2 avec 3
axes[1].scatter(X_tsne[:, 1], X_tsne[:, 2], c=dbscan_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[1].set_title('Composante 2 vs Composante 3')
axes[1].set_xlabel('Composante 2')
axes[1].set_ylabel('Composante 3')
axes[1].grid(True)

# 3ème combinaison : Composantes 1 avec 3
axes[2].scatter(X_tsne[:, 0], X_tsne[:, 2], c=dbscan_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[2].set_title('Composante 1 vs Composante 3')
axes[2].set_xlabel('Composante 1')
axes[2].set_ylabel('Composante 3')
axes[2].grid(True)

# Affichage final
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from scipy.spatial import ConvexHull

# Dictionnaire des couleurs des saisons
saison_colors = {
    'hiver': '#1f77b4',
    'printemps': '#2ca02c',
    'été': '#ff7f0e',
    'automne': '#d62728'
}

# Appliquer DBSCAN avec les paramètres spécifiés
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_TSNE_labels = dbscan.fit_predict(X_tsne)

# Récupérer les saisons en minuscules
saisons = data_filtered["Saison"].str.lower().values

# 📊 Visualisation avec t-SNE
plt.figure(figsize=(10, 8))

# Affichage des clusters
for label in np.unique(dbscan_TSNE_labels):
    cluster_mask = (dbscan_TSNE_labels == label)
    cluster_points = X_tsne[cluster_mask]
    cluster_saisons = saisons[cluster_mask]

    # Points colorés par saison dans le cluster
    for saison in np.unique(cluster_saisons):
        saison_mask = (cluster_saisons == saison)
        plt.scatter(
            cluster_points[saison_mask, 0],
            cluster_points[saison_mask, 1],
            color=saison_colors[saison],
            label=f'Cluster {label} - {saison.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # Si c'est du bruit, on marque un point noir
    if label == -1:
        plt.scatter(cluster_points[:, 0], cluster_points[:, 1], color='black', label='Bruit', edgecolors='k', s=60, alpha=0.3)

# Ajouter les titres et les axes
plt.title('Visualisation des clusters avec t-SNE et DBSCAN, coloré par saison')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from scipy.spatial import ConvexHull
import unidecode

# 🎨 Dictionnaire des couleurs des natures
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 🔤 Fonction de normalisation
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

# ✔ Nettoyage des valeurs
natures_raw = data_filtered["Nature"].str.strip().str.lower()
natures = natures_raw.apply(normalize_nature)
natures = natures.replace({
    'accidentel': 'accidentelle',
    'involontaire(particulier)': 'involontaire (particulier)',
    'involontaire ( particulier )': 'involontaire (particulier)',
    'involontaire(travaux)': 'involontaire (travaux)',
    'malveillance': 'malveillance',
    'naturelle': 'naturelle'
})

# 🤖 DBSCAN sur t-SNE
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_TSNE_labels = dbscan.fit_predict(X_tsne)

# 📊 Visualisation
plt.figure(figsize=(10, 8))

for label in np.unique(dbscan_TSNE_labels):
    cluster_mask = (dbscan_TSNE_labels == label)
    cluster_points = X_tsne[cluster_mask]
    cluster_natures = natures[cluster_mask]

    for nature in np.unique(cluster_natures):
        nature_mask = (cluster_natures == nature)
        plt.scatter(
            cluster_points[nature_mask, 0],
            cluster_points[nature_mask, 1],
            color=nature_colors.get(nature, '#999999'),
            label=f'Cluster {label} - {nature.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    if label == -1:
        plt.scatter(cluster_points[:, 0], cluster_points[:, 1], color='black', label='Bruit', edgecolors='k', s=60, alpha=0.3)

# 🎯 Légende simplifiée par nature
handles = [
    plt.Line2D([0], [0], marker='o', color='w', label=nature.capitalize(),
               markerfacecolor=color, markeredgecolor='k', markersize=10)
    for nature, color in nature_colors.items()
]
handles.append(plt.Line2D([0], [0], marker='o', color='w', label='Bruit',
                          markerfacecolor='black', markeredgecolor='k', markersize=10))

plt.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title('Visualisation des clusters DBSCAN sur t-SNE, coloré par nature')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Créer un DataFrame avec les labels de KMeans et les saisons
data_for_test = pd.DataFrame({
    'Cluster': dbscan_TSNE_labels,  # Les labels des clusters KMeans
    'Saison': data_filtered['Saison'].str.lower()  # Les saisons (en minuscule)
})

# Créer une table de contingence (comptage des saisons par cluster)
contingency_table = pd.crosstab(data_for_test['Cluster'], data_for_test['Saison'])

# Afficher la table de contingence
print("Table de contingence des saisons par cluster :")
print(contingency_table)

# Appliquer le test de Chi-Carré
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Afficher les résultats du test
print(f"\nRésultats du test de Chi-Carré:")
print(f"Statistique Chi-Carré = {chi2}")
print(f"Degrés de liberté = {dof}")
print(f"p-value = {p}")

# Interprétation des résultats
alpha = 0.05  # Niveau de signification

if p < alpha:
    print("Nous rejetons H0. La distribution des saisons est significativement différente entre les clusters.")
else:
    print("Nous ne rejetons pas H0. Il n'y a pas de différence significative dans la répartition des saisons entre les clusters.")

---
##### 3.5.4 Spectral clustering

---
###### 3.5.4.1 PCA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import SpectralClustering
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MinMaxScaler

# Appliquer MinMaxScaler pour normaliser les données
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)  # Remplace X par tes données brutes

# Réduction de la dimensionnalité avec PCA (réduire à 23 dimensions)
pca = PCA(n_components=20)
X_pca = pca.fit_transform(X_scaled)

# Calculer le Silhouette Score pour différents nombres de clusters
sil_scores = []
db_scores = []
for k in range(2, 31):  # Teste de 2 à 10 clusters
    spectral = SpectralClustering(n_clusters=k, random_state=42, affinity='nearest_neighbors')
    spectral_labels = spectral.fit_predict(X_pca)  # Appliquer Spectral Clustering
    sil_scores.append(silhouette_score(X_pca, spectral_labels))  # Calcul du Silhouette Score
    db_scores.append(davies_bouldin_score(X_pca, spectral_labels))

# Tracer le Silhouette Score
plt.figure(figsize=(10, 6))
plt.plot(range(2, 31), sil_scores, marker='o', color='g', linestyle='--')
plt.title('Silhouette Score pour Spectral Clustering avec PCA')
plt.xlabel('Nombre de clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

# 📈 Davies-Bouldin Index
plt.figure(figsize=(10, 6))
plt.plot(range(2, 31), db_scores, marker='o', linestyle='--', color='orange')
plt.title('📉 Davies-Bouldin Index par nombre de clusters')
plt.xlabel('Nombre de clusters')
plt.ylabel("Davies-Bouldin Index (↓ mieux)")
plt.grid(True)
plt.show()

# Nombre optimal de clusters basé sur le Silhouette Score
optimal_k = sil_scores.index(max(sil_scores)) + 2  # Les indices de sil_scores commencent à 2
print(f"Nombre optimal de clusters selon le Silhouette Score : {optimal_k}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import SpectralClustering
from sklearn.decomposition import PCA

# 🧠 Étape 1 : Clustering sur les 20 composantes
spectral = SpectralClustering(n_clusters=18, affinity='nearest_neighbors', random_state=42)
spectral_PCA_labels = spectral.fit_predict(X_pca)  # X_pca a 20 composantes

# 🎯 Étape 2 : Projection pour visualisation (2D)
# Refaire un PCA uniquement pour l'affichage, en gardant 2 composantes les plus "visuellement parlantes"
pca = PCA(n_components=20)
X_pca = pca.fit_transform(X_pca)  # → X_visu = données projetées pour affichage

# 🎨 Étape 3 : Affichage
plt.figure(figsize=(10, 8))
colors = plt.cm.tab10(np.linspace(0, 1, len(set(spectral_PCA_labels))))

for label, color in zip(np.unique(spectral_PCA_labels), colors):
    cluster_points = X_pca[spectral_PCA_labels == label]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1],
                color=color, label=f"Cluster {label}", edgecolors='k', s=60, alpha=0.7)

# 📌 Finalisation
plt.title("Spectral Clustering sur 20 composantes PCA — Affichage optimisé")
plt.xlabel("Composante principale 1 (visualisation)")
plt.ylabel("Composante principale 2 (visualisation)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import folium
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.cm as cm

# Assure-toi que `kmeans_PCA_labels` est déjà défini
# Ici, kmeans_PCA_labels = kmeans.labels_

# Créer une carte centrée sur la France
map_france = folium.Map(location=[46.603354, 1.888334], zoom_start=6)  # Coordonnées géographiques approximatives de la France

# Générer 16 couleurs distinctes en utilisant matplotlib
colors = cm.get_cmap('tab20', 18)  # 'tab20' génère 20 couleurs distinctes
cluster_colors = [colors(i) for i in range(18)]  # Attribuer une couleur à chaque cluster

# Afficher les points des clusters sur la carte
for i, row in data_filtered.iterrows():
    # Limiter les valeurs des coordonnées pour s'assurer qu'elles sont dans les bornes de la France
    latitude = np.clip(row['latitude'], 42, 51)
    longitude = np.clip(row['longitude'], -5, 10)

    # Utiliser kmeans_PCA_labels pour récupérer le numéro du cluster pour chaque point
    cluster_label = spectral_PCA_labels[i]

    # Attribuer une couleur en fonction du cluster
    cluster_color = cluster_colors[cluster_label]

    # Créer un popup avec le numéro du cluster
    popup = folium.Popup(f'Cluster: {cluster_label}', parse_html=True)

    folium.CircleMarker(
        location=[latitude, longitude],
        radius=5,
        color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill=True,
        fill_color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill_opacity=0.6,
        popup=popup  # Ajouter le popup ici
    ).add_to(map_france)

# Afficher la carte dans un fichier HTML
map_france.save('../results_img_csv/map_clusters_spectral_PCA.html')

# Afficher un message de succès
print("✅ Carte avec les points des clusters et leurs numéros enregistrée sous 'map_clusters_spectral_PCA.html'.")

In [ ]:
from sklearn.cluster import SpectralClustering
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
import numpy as np

# 🎨 Couleurs des saisons
saison_colors = {
    'hiver': '#1f77b4',
    'printemps': '#2ca02c',
    'été': '#ff7f0e',
    'automne': '#d62728'
}

# ✅ Utilise uniquement les 2 premières composantes pour la visualisation
# 🧠 Clustering spectral
spectral_best = SpectralClustering(n_clusters=18, random_state=42, affinity='nearest_neighbors')
spectral_PCA_labels = spectral_best.fit_predict(X_pca)

# 📉 Récupère les saisons
saisons = data_filtered["Saison"].str.lower().values

# 📊 Plot
plt.figure(figsize=(10, 8))

for cluster_id in np.unique(spectral_PCA_labels):
    cluster_mask = (spectral_PCA_labels == cluster_id)
    cluster_points = X_pca[cluster_mask]
    cluster_saisons = saisons[cluster_mask]

    for saison in np.unique(cluster_saisons):
        saison_mask = (cluster_saisons == saison)
        plt.scatter(
            cluster_points[saison_mask, 0],
            cluster_points[saison_mask, 1],
            color=saison_colors.get(saison, '#aaaaaa'),
            label=f'Cluster {cluster_id} - {saison.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # Centre approx
    if len(cluster_points) > 0:
        center = cluster_points.mean(axis=0)
        plt.text(center[0], center[1], str(cluster_id),
                 fontsize=12, fontweight='bold', color='black',
                 ha='center', va='center',
                 bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle'))

# 🧼 Esthétique
plt.grid(True, linestyle='--', alpha=0.5)
plt.title("Spectral Clustering (2D PCA) – Couleurs par saison")
plt.xlabel("Composante principale 1")
plt.ylabel("Composante principale 2")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import SpectralClustering
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
import numpy as np
import unidecode

# 🎨 Dictionnaire des couleurs pour chaque nature
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 🔤 Fonction de normalisation
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

# ✔ Nettoyage des valeurs
natures_raw = data_filtered["Nature"].str.strip().str.lower()
natures = natures_raw.apply(normalize_nature)
natures = natures.replace({
    'accidentel': 'accidentelle',
    'involontaire(particulier)': 'involontaire (particulier)',
    'involontaire ( particulier )': 'involontaire (particulier)',
    'involontaire(travaux)': 'involontaire (travaux)',
    'malveillance': 'malveillance',
    'naturelle': 'naturelle'
})

# 🤖 Spectral Clustering sur PCA
spectral_best = SpectralClustering(n_clusters=18, random_state=42, affinity='nearest_neighbors')
spectral_PCA_labels = spectral_best.fit_predict(X_pca)

X_pca_visu = X_pca[:, :2]
centroids_2D = centroids[:, :2]
# 📊 Visualisation
plt.figure(figsize=(10, 8))

for cluster_id in np.unique(spectral_PCA_labels):
    cluster_mask = (spectral_PCA_labels == cluster_id)
    cluster_points = X_pca_visu[cluster_mask]
    cluster_natures = natures[cluster_mask]

    # 🎨 Points du cluster, colorés par nature
    for nature in np.unique(cluster_natures):
        nature_mask = (cluster_natures == nature)
        plt.scatter(
            cluster_points[nature_mask, 0],
            cluster_points[nature_mask, 1],
            color=nature_colors.get(nature, '#999999'),
            label=f'Cluster {cluster_id} - {nature.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # 🧭 Affichage du numéro du cluster (centre approx.)
    cluster_center = cluster_points.mean(axis=0)
    plt.text(
        cluster_center[0], cluster_center[1],
        str(cluster_id), fontsize=14, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# ➕ Légende simplifiée
handles = [
    plt.Line2D([0], [0], marker='o', color='w', label=nature.capitalize(),
               markerfacecolor=color, markeredgecolor='k', markersize=10)
    for nature, color in nature_colors.items()
]

plt.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title('Clustering Spectral (PCA) - Coloration par nature')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Créer un DataFrame avec les labels de KMeans et les saisons
data_for_test = pd.DataFrame({
    'Cluster': spectral_PCA_labels,  # Les labels des clusters KMeans
    'Saison': data_filtered['Saison'].str.lower()  # Les saisons (en minuscule)
})

# Créer une table de contingence (comptage des saisons par cluster)
contingency_table = pd.crosstab(data_for_test['Cluster'], data_for_test['Saison'])

# Afficher la table de contingence
print("Table de contingence des saisons par cluster :")
print(contingency_table)

# Appliquer le test de Chi-Carré
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Afficher les résultats du test
print(f"\nRésultats du test de Chi-Carré:")
print(f"Statistique Chi-Carré = {chi2}")
print(f"Degrés de liberté = {dof}")
print(f"p-value = {p}")

# Interprétation des résultats
alpha = 0.05  # Niveau de signification

if p < alpha:
    print("Nous rejetons H0. La distribution des saisons est significativement différente entre les clusters.")
else:
    print("Nous ne rejetons pas H0. Il n'y a pas de différence significative dans la répartition des saisons entre les clusters.")

---
###### 3.5.4.2 TSNE

In [ ]:
tsne = TSNE(n_components=3, random_state=42, perplexity=100)
X_tsne = tsne.fit_transform(X_scaled)

# Calculer le Silhouette Score pour différents nombres de clusters
sil_scores = []
db_scores = []
for k in range(2, 31):  # Teste de 2 à 10 clusters
    spectral = SpectralClustering(n_clusters=k, random_state=42, affinity='nearest_neighbors')
    spectral_labels = spectral.fit_predict(X_tsne)  # Appliquer Agglomerative Clustering
    sil_scores.append(silhouette_score(X_tsne, spectral_labels))  # Calcul du Silhouette Score
    db_scores.append(davies_bouldin_score(X_tsne, spectral_labels))

# Tracer le Silhouette Score
plt.figure(figsize=(10, 6))
plt.plot(range(2, 31), sil_scores, marker='o', color='g', linestyle='--')
plt.title('Silhouette Score pour Agglomerative Clustering avec PCA')
plt.xlabel('Nombre de clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

# 📈 Davies-Bouldin Index
plt.figure(figsize=(10, 6))
plt.plot(range(2, 31), db_scores, marker='o', linestyle='--', color='orange')
plt.title('📉 Davies-Bouldin Index par nombre de clusters')
plt.xlabel('Nombre de clusters')
plt.ylabel("Davies-Bouldin Index (↓ mieux)")
plt.grid(True)
plt.show()

# Nombre optimal de clusters basé sur le Silhouette Score
optimal_k = sil_scores.index(max(sil_scores)) + 2  # Les indices de sil_scores commencent à 2
print(f"Nombre optimal de clusters selon le Silhouette Score : {optimal_k}")

In [ ]:
from sklearn.cluster import SpectralClustering
import matplotlib.pyplot as plt
import numpy as np

# Appliquer Spectral Clustering avec les meilleurs hyperparamètres trouvés
spectral = SpectralClustering(n_clusters=2, affinity='nearest_neighbors', random_state=42)
spectral_TSNE_labels = spectral.fit_predict(X_tsne)

# 📊 Visualisation avec différentes combinaisons de composantes t-SNE
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# 1ère combinaison : Composantes 1 avec 2
axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=spectral_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[0].set_title('Composante 1 vs Composante 2')
axes[0].set_xlabel('Composante 1')
axes[0].set_ylabel('Composante 2')
axes[0].grid(True)

# 2ème combinaison : Composantes 2 avec 3
axes[1].scatter(X_tsne[:, 1], X_tsne[:, 2], c=spectral_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[1].set_title('Composante 2 vs Composante 3')
axes[1].set_xlabel('Composante 2')
axes[1].set_ylabel('Composante 3')
axes[1].grid(True)

# 3ème combinaison : Composantes 1 avec 3
axes[2].scatter(X_tsne[:, 0], X_tsne[:, 2], c=spectral_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[2].set_title('Composante 1 vs Composante 3')
axes[2].set_xlabel('Composante 1')
axes[2].set_ylabel('Composante 3')
axes[2].grid(True)

# Affichage final
plt.tight_layout()
plt.show()

In [ ]:
import folium
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.cm as cm

# Assure-toi que `kmeans_PCA_labels` est déjà défini
# Ici, kmeans_PCA_labels = kmeans.labels_

# Créer une carte centrée sur la France
map_france = folium.Map(location=[46.603354, 1.888334], zoom_start=6)  # Coordonnées géographiques approximatives de la France

# Générer 16 couleurs distinctes en utilisant matplotlib
colors = cm.get_cmap('tab20', 18)  # 'tab20' génère 20 couleurs distinctes
cluster_colors = [colors(i) for i in range(18)]  # Attribuer une couleur à chaque cluster

# Afficher les points des clusters sur la carte
for i, row in data_filtered.iterrows():
    # Limiter les valeurs des coordonnées pour s'assurer qu'elles sont dans les bornes de la France
    latitude = np.clip(row['latitude'], 42, 51)
    longitude = np.clip(row['longitude'], -5, 10)

    # Utiliser kmeans_PCA_labels pour récupérer le numéro du cluster pour chaque point
    cluster_label = spectral_TSNE_labels[i]

    # Attribuer une couleur en fonction du cluster
    cluster_color = cluster_colors[cluster_label]

    # Créer un popup avec le numéro du cluster
    popup = folium.Popup(f'Cluster: {cluster_label}', parse_html=True)

    folium.CircleMarker(
        location=[latitude, longitude],
        radius=5,
        color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill=True,
        fill_color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill_opacity=0.6,
        popup=popup  # Ajouter le popup ici
    ).add_to(map_france)

# Afficher la carte dans un fichier HTML
map_france.save('../results_img_csv/map_clusters_spectral_TSNE.html')

# Afficher un message de succès
print("✅ Carte avec les points des clusters et leurs numéros enregistrée sous 'map_clusters_spectral_TSNE.html'.")

In [ ]:
from sklearn.cluster import SpectralClustering
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
import numpy as np

# 🎨 Dictionnaire des couleurs pour les saisons
saison_colors = {
    'hiver': '#1f77b4',
    'printemps': '#2ca02c',
    'été': '#ff7f0e',
    'automne': '#d62728'
}

# 🔍 Appliquer Spectral Clustering
spectral = SpectralClustering(n_clusters=2, affinity='nearest_neighbors', random_state=42)
spectral_TSNE_labels = spectral.fit_predict(X_tsne)

# ⬇️ Récupérer les saisons en minuscules
saisons = data_filtered["Saison"].str.lower().values

# 📊 Visualisation
plt.figure(figsize=(10, 8))

for cluster_id in np.unique(spectral_TSNE_labels):
    cluster_mask = (spectral_TSNE_labels == cluster_id)
    cluster_points = X_tsne[cluster_mask]
    cluster_saisons = saisons[cluster_mask]

    # 🎨 Points colorés par saison dans le cluster
    for saison in np.unique(cluster_saisons):
        saison_mask = (cluster_saisons == saison)
        plt.scatter(
            cluster_points[saison_mask, 0],
            cluster_points[saison_mask, 1],
            color=saison_colors[saison],
            label=f'Cluster {cluster_id} - {saison.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # 🔢 Numéro du cluster (au centre approximatif)
    cluster_center = cluster_points.mean(axis=0)
    plt.text(
        cluster_center[0], cluster_center[1],
        str(cluster_id), fontsize=14, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# Finalisation du graphique
plt.title('Clustering Spectral (t-SNE) - Coloration par saison')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import SpectralClustering
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
import numpy as np
import unidecode

# 🎨 Dictionnaire des couleurs pour chaque nature
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 🔤 Fonction pour normaliser les natures
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

# ✔ Nettoyage des valeurs
natures_raw = data_filtered["Nature"].str.strip().str.lower()
natures = natures_raw.apply(normalize_nature)
natures = natures.replace({
    'accidentel': 'accidentelle',
    'involontaire(particulier)': 'involontaire (particulier)',
    'involontaire ( particulier )': 'involontaire (particulier)',
    'involontaire(travaux)': 'involontaire (travaux)',
    'malveillance': 'malveillance',
    'naturelle': 'naturelle'
})

# 🤖 Spectral Clustering sur t-SNE
spectral = SpectralClustering(n_clusters=2, affinity='nearest_neighbors', random_state=42)
spectral_TSNE_labels = spectral.fit_predict(X_tsne)

# 📊 Visualisation
plt.figure(figsize=(10, 8))

for cluster_id in np.unique(spectral_TSNE_labels):
    cluster_mask = (spectral_TSNE_labels == cluster_id)
    cluster_points = X_tsne[cluster_mask]
    cluster_natures = natures[cluster_mask]

    # 🎨 Points du cluster, colorés par nature
    for nature in np.unique(cluster_natures):
        nature_mask = (cluster_natures == nature)
        plt.scatter(
            cluster_points[nature_mask, 0],
            cluster_points[nature_mask, 1],
            color=nature_colors.get(nature, '#999999'),
            label=f'Cluster {cluster_id} - {nature.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # 🧭 Numéro du cluster
    cluster_center = cluster_points.mean(axis=0)
    plt.text(
        cluster_center[0], cluster_center[1],
        str(cluster_id), fontsize=14, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# ➕ Légende simplifiée
handles = [
    plt.Line2D([0], [0], marker='o', color='w', label=nature.capitalize(),
               markerfacecolor=color, markeredgecolor='k', markersize=10)
    for nature, color in nature_colors.items()
]

plt.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title('Clustering Spectral (t-SNE) - Coloration par nature')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Créer un DataFrame avec les labels de KMeans et les saisons
data_for_test = pd.DataFrame({
    'Cluster': spectral_TSNE_labels,  # Les labels des clusters KMeans
    'Saison': data_filtered['Saison'].str.lower()  # Les saisons (en minuscule)
})

# Créer une table de contingence (comptage des saisons par cluster)
contingency_table = pd.crosstab(data_for_test['Cluster'], data_for_test['Saison'])

# Afficher la table de contingence
print("Table de contingence des saisons par cluster :")
print(contingency_table)

# Appliquer le test de Chi-Carré
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Afficher les résultats du test
print(f"\nRésultats du test de Chi-Carré:")
print(f"Statistique Chi-Carré = {chi2}")
print(f"Degrés de liberté = {dof}")
print(f"p-value = {p}")

# Interprétation des résultats
alpha = 0.05  # Niveau de signification

if p < alpha:
    print("Nous rejetons H0. La distribution des saisons est significativement différente entre les clusters.")
else:
    print("Nous ne rejetons pas H0. Il n'y a pas de différence significative dans la répartition des saisons entre les clusters.")

---
##### 3.5.5 Agglomerative Clustering

---
###### 3.5.5.1 PCA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Appliquer MinMaxScaler pour normaliser les données
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)  # Remplace X par tes données brutes

# Réduction de la dimensionnalité avec PCA (réduire à 46 dimensions)
pca = PCA(n_components=20)
X_pca = pca.fit_transform(X_scaled)

# Calculer le Silhouette Score pour différents nombres de clusters
sil_scores = []
db_scores = []
for k in range(2, 31):  # Teste de 2 à 10 clusters
    agglo = AgglomerativeClustering(n_clusters=k, linkage="ward")  # Utilisation de la méthode de Ward
    agglo_labels = agglo.fit_predict(X_pca)  # Appliquer Agglomerative Clustering
    sil_scores.append(silhouette_score(X_pca, agglo_labels))  # Calcul du Silhouette Score
    db_scores.append(davies_bouldin_score(X_pca, agglo_labels))

# Tracer le Silhouette Score
plt.figure(figsize=(10, 6))
plt.plot(range(2, 31), sil_scores, marker='o', color='g', linestyle='--')
plt.title('Silhouette Score pour Agglomerative Clustering avec PCA')
plt.xlabel('Nombre de clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

# 📈 Davies-Bouldin Index
plt.figure(figsize=(10, 6))
plt.plot(range(2, 31), db_scores, marker='o', linestyle='--', color='orange')
plt.title('📉 Davies-Bouldin Index par nombre de clusters')
plt.xlabel('Nombre de clusters')
plt.ylabel("Davies-Bouldin Index (↓ mieux)")
plt.grid(True)
plt.show()

# Nombre optimal de clusters basé sur le Silhouette Score
optimal_k = sil_scores.index(max(sil_scores)) + 2  # Les indices de sil_scores commencent à 2
print(f"Nombre optimal de clusters selon le Silhouette Score : {optimal_k}")

In [ ]:
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
import numpy as np

# Appliquer Agglomerative Clustering avec les meilleurs hyperparamètres trouvés
agglomerative_clus = AgglomerativeClustering(n_clusters=17, linkage='ward')
agglo_clus_PCA_labels = agglomerative_clus.fit_predict(X_pca)

X_pca_visu = X_pca[:, :2]
centroids_2D = centroids[:, :2]
# Visualisation des clusters
plt.figure(figsize=(10, 8))

# Affichage des clusters
unique_labels = set(agglo_clus_PCA_labels)
colors = plt.cm.plasma(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, colors):
    # Points du cluster
    cluster_points = X_pca_visu[agglo_clus_PCA_labels == label]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1], color=color, label=f"Cluster {label}", edgecolors='k', alpha=0.7, s=50)

    # Tracer les contours pour les clusters (convex hull)
    if len(cluster_points) >= 3:
        hull = ConvexHull(cluster_points)
        for simplex in hull.simplices:
            plt.plot(cluster_points[simplex, 0], cluster_points[simplex, 1], 'k--', linewidth=1.2)

# Ajouter les titres et les axes
plt.title('Visualisation des clusters avec Agglomerative Clustering')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.grid(True)
plt.colorbar(label="Cluster")
plt.legend()
plt.show()

In [ ]:
import folium
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.cm as cm

# Assure-toi que `kmeans_PCA_labels` est déjà défini
# Ici, kmeans_PCA_labels = kmeans.labels_

# Créer une carte centrée sur la France
map_france = folium.Map(location=[46.603354, 1.888334], zoom_start=6)  # Coordonnées géographiques approximatives de la France

# Générer 16 couleurs distinctes en utilisant matplotlib
colors = cm.get_cmap('tab20', 18)  # 'tab20' génère 20 couleurs distinctes
cluster_colors = [colors(i) for i in range(18)]  # Attribuer une couleur à chaque cluster

# Afficher les points des clusters sur la carte
for i, row in data_filtered.iterrows():
    # Limiter les valeurs des coordonnées pour s'assurer qu'elles sont dans les bornes de la France
    latitude = np.clip(row['latitude'], 42, 51)
    longitude = np.clip(row['longitude'], -5, 10)

    # Utiliser kmeans_PCA_labels pour récupérer le numéro du cluster pour chaque point
    cluster_label = agglo_clus_PCA_labels[i]

    # Attribuer une couleur en fonction du cluster
    cluster_color = cluster_colors[cluster_label]

    # Créer un popup avec le numéro du cluster
    popup = folium.Popup(f'Cluster: {cluster_label}', parse_html=True)

    folium.CircleMarker(
        location=[latitude, longitude],
        radius=5,
        color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill=True,
        fill_color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill_opacity=0.6,
        popup=popup  # Ajouter le popup ici
    ).add_to(map_france)

# Afficher la carte dans un fichier HTML
map_france.save('../results_img_csv/map_clusters_agglo_PCA.html')

# Afficher un message de succès
print("✅ Carte avec les points des clusters et leurs numéros enregistrée sous 'map_clusters_agglo_PCA.html'.")

In [ ]:
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
import numpy as np

# 🎨 Couleurs des saisons
saison_colors = {
    'hiver': '#1f77b4',
    'printemps': '#2ca02c',
    'été': '#ff7f0e',
    'automne': '#d62728'
}

# 🔗 Appliquer Agglomerative Clustering
agglomerative_clus = AgglomerativeClustering(n_clusters=17, linkage='ward')
agglo_clus_PCA_labels = agglomerative_clus.fit_predict(X_pca)

X_pca_visu = X_pca[:, :2]
centroids_2D = centroids[:, :2]
# ⬇️ Récupérer les saisons
saisons = data_filtered["Saison"].str.lower().values

# 📊 Visualisation
plt.figure(figsize=(10, 8))

for cluster_id in np.unique(agglo_clus_PCA_labels):
    cluster_mask = (agglo_clus_PCA_labels == cluster_id)
    cluster_points = X_pca_visu[cluster_mask]
    cluster_saisons = saisons[cluster_mask]

    # Points colorés par saison
    for saison in np.unique(cluster_saisons):
        saison_mask = (cluster_saisons == saison)
        plt.scatter(
            cluster_points[saison_mask, 0],
            cluster_points[saison_mask, 1],
            color=saison_colors[saison],
            label=f'Cluster {cluster_id} - {saison.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # ➕ Contours avec ConvexHull
    if len(cluster_points) >= 3:
        hull = ConvexHull(cluster_points)
        for simplex in hull.simplices:
            plt.plot(cluster_points[simplex, 0], cluster_points[simplex, 1], 'k--', linewidth=1)

    # 🔢 Numéro du cluster
    cluster_center = cluster_points.mean(axis=0)
    plt.text(
        cluster_center[0], cluster_center[1],
        str(cluster_id), fontsize=14, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# Final touches
plt.title('Clustering Agglomeratif (PCA) - Coloration par saison')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
import numpy as np
import unidecode

# 🎨 Couleurs des natures
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 🔤 Nettoyage des labels "Nature"
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

natures_raw = data_filtered["Nature"].str.strip().str.lower()
natures = natures_raw.apply(normalize_nature)
natures = natures.replace({
    'accidentel': 'accidentelle',
    'involontaire(particulier)': 'involontaire (particulier)',
    'involontaire ( particulier )': 'involontaire (particulier)',
    'involontaire(travaux)': 'involontaire (travaux)',
    'malveillance': 'malveillance',
    'naturelle': 'naturelle'
})

# 🤖 Clustering Agglomératif
agglomerative_clus = AgglomerativeClustering(n_clusters=17, linkage='ward')
agglo_clus_PCA_labels = agglomerative_clus.fit_predict(X_pca)

X_pca_visu = X_pca[:, :2]
centroids_2D = centroids[:, :2]
# 📊 Visualisation
plt.figure(figsize=(10, 8))

for cluster_id in np.unique(agglo_clus_PCA_labels):
    cluster_mask = (agglo_clus_PCA_labels == cluster_id)
    cluster_points = X_pca_visu[cluster_mask]
    cluster_natures = natures[cluster_mask]

    # 🔸 Points colorés par nature
    for nature in np.unique(cluster_natures):
        nature_mask = (cluster_natures == nature)
        plt.scatter(
            cluster_points[nature_mask, 0],
            cluster_points[nature_mask, 1],
            color=nature_colors.get(nature, '#999999'),
            label=f'Cluster {cluster_id} - {nature.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # ➕ Tracer les contours
    if len(cluster_points) >= 3:
        hull = ConvexHull(cluster_points)
        for simplex in hull.simplices:
            plt.plot(cluster_points[simplex, 0], cluster_points[simplex, 1], 'k--', linewidth=1)

    # 🔢 Numéro du cluster
    cluster_center = cluster_points.mean(axis=0)
    plt.text(
        cluster_center[0], cluster_center[1],
        str(cluster_id), fontsize=14, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# ➕ Légende claire
handles = [
    plt.Line2D([0], [0], marker='o', color='w', label=nature.capitalize(),
               markerfacecolor=color, markeredgecolor='k', markersize=10)
    for nature, color in nature_colors.items()
]

plt.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title('Clustering Agglomératif (PCA) - Coloration par nature')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Créer un DataFrame avec les labels de KMeans et les saisons
data_for_test = pd.DataFrame({
    'Cluster': agglo_clus_PCA_labels,  # Les labels des clusters KMeans
    'Saison': data_filtered['Saison'].str.lower()  # Les saisons (en minuscule)
})

# Créer une table de contingence (comptage des saisons par cluster)
contingency_table = pd.crosstab(data_for_test['Cluster'], data_for_test['Saison'])

# Afficher la table de contingence
print("Table de contingence des saisons par cluster :")
print(contingency_table)

# Appliquer le test de Chi-Carré
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Afficher les résultats du test
print(f"\nRésultats du test de Chi-Carré:")
print(f"Statistique Chi-Carré = {chi2}")
print(f"Degrés de liberté = {dof}")
print(f"p-value = {p}")

# Interprétation des résultats
alpha = 0.05  # Niveau de signification

if p < alpha:
    print("Nous rejetons H0. La distribution des saisons est significativement différente entre les clusters.")
else:
    print("Nous ne rejetons pas H0. Il n'y a pas de différence significative dans la répartition des saisons entre les clusters.")

---
###### 3.5.5.2 TSNE

In [ ]:
tsne = TSNE(n_components=3, random_state=42, perplexity=100)
X_tsne = tsne.fit_transform(X_scaled)

# Calculer le Silhouette Score pour différents nombres de clusters
sil_scores = []
db_scores = []
for k in range(2, 31):  # Teste de 2 à 10 clusters
    agglo = AgglomerativeClustering(n_clusters=k, linkage="ward")  # Utilisation de la méthode de Ward
    agglo_labels = agglo.fit_predict(X_tsne)  # Appliquer Agglomerative Clustering
    sil_scores.append(silhouette_score(X_tsne, agglo_labels))  # Calcul du Silhouette Score
    db_scores.append(davies_bouldin_score(X_tsne, agglo_labels))

# Tracer le Silhouette Score
plt.figure(figsize=(10, 6))
plt.plot(range(2, 31), sil_scores, marker='o', color='g', linestyle='--')
plt.title('Silhouette Score pour Agglomerative Clustering avec PCA')
plt.xlabel('Nombre de clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

# 📈 Davies-Bouldin Index
plt.figure(figsize=(10, 6))
plt.plot(range(2, 31), db_scores, marker='o', linestyle='--', color='orange')
plt.title('📉 Davies-Bouldin Index par nombre de clusters')
plt.xlabel('Nombre de clusters')
plt.ylabel("Davies-Bouldin Index (↓ mieux)")
plt.grid(True)
plt.show()

# Nombre optimal de clusters basé sur le Silhouette Score
optimal_k = sil_scores.index(max(sil_scores)) + 2  # Les indices de sil_scores commencent à 2
print(f"Nombre optimal de clusters selon le Silhouette Score : {optimal_k}")

In [ ]:
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
import numpy as np

# Appliquer Agglomerative Clustering avec les meilleurs hyperparamètres trouvés
agglomerative_clustering = AgglomerativeClustering(n_clusters=14, linkage='ward')
agglo_clus_TSNE_labels = agglomerative_clustering.fit_predict(X_tsne)

# 📊 Visualisation avec différentes combinaisons de composantes t-SNE
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# 1ère combinaison : Composantes 1 avec 2
axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=agglo_clus_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[0].set_title('Composante 1 vs Composante 2')
axes[0].set_xlabel('Composante 1')
axes[0].set_ylabel('Composante 2')
axes[0].grid(True)

# 2ème combinaison : Composantes 2 avec 3
axes[1].scatter(X_tsne[:, 1], X_tsne[:, 2], c=agglo_clus_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[1].set_title('Composante 2 vs Composante 3')
axes[1].set_xlabel('Composante 2')
axes[1].set_ylabel('Composante 3')
axes[1].grid(True)

# 3ème combinaison : Composantes 1 avec 3
axes[2].scatter(X_tsne[:, 0], X_tsne[:, 2], c=agglo_clus_TSNE_labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[2].set_title('Composante 1 vs Composante 3')
axes[2].set_xlabel('Composante 1')
axes[2].set_ylabel('Composante 3')
axes[2].grid(True)

# Affichage final
plt.tight_layout()
plt.show()

In [ ]:
import folium
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.cm as cm

# Assure-toi que `kmeans_PCA_labels` est déjà défini
# Ici, kmeans_PCA_labels = kmeans.labels_

# Créer une carte centrée sur la France
map_france = folium.Map(location=[46.603354, 1.888334], zoom_start=6)  # Coordonnées géographiques approximatives de la France

# Générer 16 couleurs distinctes en utilisant matplotlib
colors = cm.get_cmap('tab20', 18)  # 'tab20' génère 20 couleurs distinctes
cluster_colors = [colors(i) for i in range(18)]  # Attribuer une couleur à chaque cluster

# Afficher les points des clusters sur la carte
for i, row in data_filtered.iterrows():
    # Limiter les valeurs des coordonnées pour s'assurer qu'elles sont dans les bornes de la France
    latitude = np.clip(row['latitude'], 42, 51)
    longitude = np.clip(row['longitude'], -5, 10)

    # Utiliser kmeans_PCA_labels pour récupérer le numéro du cluster pour chaque point
    cluster_label = agglo_clus_TSNE_labels[i]

    # Attribuer une couleur en fonction du cluster
    cluster_color = cluster_colors[cluster_label]

    # Créer un popup avec le numéro du cluster
    popup = folium.Popup(f'Cluster: {cluster_label}', parse_html=True)

    folium.CircleMarker(
        location=[latitude, longitude],
        radius=5,
        color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill=True,
        fill_color=f'#{int(cluster_color[0]*255):02x}{int(cluster_color[1]*255):02x}{int(cluster_color[2]*255):02x}',
        fill_opacity=0.6,
        popup=popup  # Ajouter le popup ici
    ).add_to(map_france)

# Afficher la carte dans un fichier HTML
map_france.save('../results_img_csv/map_clusters_agglo_TSNE.html')

# Afficher un message de succès
print("✅ Carte avec les points des clusters et leurs numéros enregistrée sous 'map_clusters_agglo_TSNE.html'.")

In [ ]:
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
import numpy as np

# 🎨 Couleurs par saison
saison_colors = {
    'hiver': '#1f77b4',
    'printemps': '#2ca02c',
    'été': '#ff7f0e',
    'automne': '#d62728'
}

# 🔗 Clustering
agglomerative_clustering = AgglomerativeClustering(n_clusters=14, linkage='ward')
agglo_clus_TSNE_labels = agglomerative_clustering.fit_predict(X_tsne)

# ⬇️ Saisons associées
saisons = data_filtered["Saison"].str.lower().values

# 📊 Visualisation
plt.figure(figsize=(10, 8))

for cluster_id in np.unique(agglo_clus_TSNE_labels):
    cluster_mask = (agglo_clus_TSNE_labels == cluster_id)
    cluster_points = X_tsne[cluster_mask]
    cluster_saisons = saisons[cluster_mask]

    # 🔵 Points colorés par saison
    for saison in np.unique(cluster_saisons):
        saison_mask = (cluster_saisons == saison)
        plt.scatter(
            cluster_points[saison_mask, 0],
            cluster_points[saison_mask, 1],
            color=saison_colors[saison],
            label=f'Cluster {cluster_id} - {saison.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # 🔢 Numéro du cluster
    cluster_center = cluster_points.mean(axis=0)
    plt.text(
        cluster_center[0], cluster_center[1],
        str(cluster_id), fontsize=14, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# Final touches
plt.title('Clustering Agglomeratif (t-SNE) - Coloration par saison')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
import numpy as np
import unidecode

# 🎨 Couleurs pour chaque nature
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 🔤 Nettoyage des libellés Nature
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

natures_raw = data_filtered["Nature"].str.strip().str.lower()
natures = natures_raw.apply(normalize_nature)
natures = natures.replace({
    'accidentel': 'accidentelle',
    'involontaire(particulier)': 'involontaire (particulier)',
    'involontaire ( particulier )': 'involontaire (particulier)',
    'involontaire(travaux)': 'involontaire (travaux)',
    'malveillance': 'malveillance',
    'naturelle': 'naturelle'
})

# 🔗 Clustering Agglomératif
agglomerative_clustering = AgglomerativeClustering(n_clusters=14, linkage='ward')
agglo_clus_TSNE_labels = agglomerative_clustering.fit_predict(X_tsne)

# 📊 Visualisation
plt.figure(figsize=(10, 8))

for cluster_id in np.unique(agglo_clus_TSNE_labels):
    cluster_mask = (agglo_clus_TSNE_labels == cluster_id)
    cluster_points = X_tsne[cluster_mask]
    cluster_natures = natures[cluster_mask]

    # Points colorés par nature
    for nature in np.unique(cluster_natures):
        nature_mask = (cluster_natures == nature)
        plt.scatter(
            cluster_points[nature_mask, 0],
            cluster_points[nature_mask, 1],
            color=nature_colors.get(nature, '#999999'),
            label=f'Cluster {cluster_id} - {nature.capitalize()}',
            edgecolors='k',
            s=60,
            alpha=0.7
        )

    # 🔢 Numéro du cluster
    cluster_center = cluster_points.mean(axis=0)
    plt.text(
        cluster_center[0], cluster_center[1],
        str(cluster_id), fontsize=14, fontweight='bold',
        color='black', ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='circle')
    )

# Légende claire et compacte
handles = [
    plt.Line2D([0], [0], marker='o', color='w', label=nature.capitalize(),
               markerfacecolor=color, markeredgecolor='k', markersize=10)
    for nature, color in nature_colors.items()
]

plt.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title('Clustering Agglomératif (t-SNE) - Coloration par nature')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Créer un DataFrame avec les labels de KMeans et les saisons
data_for_test = pd.DataFrame({
    'Cluster': agglo_clus_TSNE_labels,  # Les labels des clusters KMeans
    'Saison': data_filtered['Saison'].str.lower()  # Les saisons (en minuscule)
})

# Créer une table de contingence (comptage des saisons par cluster)
contingency_table = pd.crosstab(data_for_test['Cluster'], data_for_test['Saison'])

# Afficher la table de contingence
print("Table de contingence des saisons par cluster :")
print(contingency_table)

# Appliquer le test de Chi-Carré
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Afficher les résultats du test
print(f"\nRésultats du test de Chi-Carré:")
print(f"Statistique Chi-Carré = {chi2}")
print(f"Degrés de liberté = {dof}")
print(f"p-value = {p}")

# Interprétation des résultats
alpha = 0.05  # Niveau de signification

if p < alpha:
    print("Nous rejetons H0. La distribution des saisons est significativement différente entre les clusters.")
else:
    print("Nous ne rejetons pas H0. Il n'y a pas de différence significative dans la répartition des saisons entre les clusters.")

---
##### 3.5.6 Valeurs dans les points des clusters

In [ ]:
pd.crosstab(kmeans_PCA_labels, data_filtered["Nature"])

In [ ]:
pd.crosstab(gmm_PCA_labels, data_filtered["Nature"])

In [ ]:
pd.crosstab(dbscan_PCA_labels_best, data_filtered["Nature"])

In [ ]:
pd.crosstab(spectral_PCA_labels, data_filtered["Nature"])

In [ ]:
pd.crosstab(agglo_clus_PCA_labels, data_filtered["Nature"])

---
##### 3.5.7 Résultats dans les clusters

---
###### 3.5.7.1 Kmeans

In [ ]:
# Copie pour éviter de modifier l'original
df_full_kmeans = data_filtered.copy()

# Ajout des labels de cluster
df_full_kmeans['Cluster'] = kmeans_PCA_labels

In [ ]:
# Filtrer les points du cluster 1
cluster_3_data_kmeans = df_full_kmeans[df_full_kmeans['Cluster'] == 3]

# Afficher les premières lignes pour voir ce qu’il contient
print(cluster_3_data_kmeans[['Nature', 'temp_mean', 'rhum_mean', 'prec24h_mean', 'wspd_mean']].head())

In [ ]:
# Moyennes des variables météo dans le cluster 1
print("📊 Moyennes météo - Cluster 1 :")
print(cluster_3_data_kmeans[['temp_mean', 'rhum_mean', 'prec24h_mean', 'wspd_mean']].mean().round(2))

# Répartition des natures dans ce cluster
print("\n🔥 Répartition des types de feu - Cluster 1 :")
print(cluster_3_data_kmeans['Nature'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import unidecode

# 🔄 Normalisation des natures
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

data_filtered['Nature_norm'] = data_filtered['Nature'].apply(normalize_nature)

# 🎯 Données pour histogramme
cluster_labels = kmeans_PCA_labels  # ← change selon ton clustering
df_clusters = data_filtered.copy()
df_clusters["Cluster"] = cluster_labels

# Compter les natures par cluster
nature_counts = df_clusters.groupby(['Cluster', 'Nature_norm']).size().reset_index(name='Count')
pivot_table = nature_counts.pivot(index='Cluster', columns='Nature_norm', values='Count').fillna(0)

# 🎨 Couleurs cohérentes
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 📊 Plot empilé
fig, ax = plt.subplots(figsize=(12, 7))

bottom = np.zeros(len(pivot_table))
x = np.arange(len(pivot_table))

for nature in pivot_table.columns:
    counts = pivot_table[nature].values
    bars = ax.bar(x, counts, bottom=bottom, label=nature.capitalize(), color=nature_colors.get(nature, '#999999'), edgecolor='k')

    # 🧾 Ajout des annotations dans chaque bloc
    for i, (bar, count) in enumerate(zip(bars, counts)):
        if count > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + bar.get_height() / 2,
                int(count),
                ha='center', va='center',
                fontsize=9, color='white', fontweight='bold'
            )

    bottom += counts

# 🏷️ Légendes & axes
ax.set_title("Répartition des natures par cluster (avec nombre d'incendies)")
ax.set_xlabel("Cluster")
ax.set_ylabel("Nombre d'incendies")
ax.set_xticks(x)
ax.set_xticklabels(pivot_table.index)
ax.legend(title="Nature", bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import unidecode

# 🔄 Normalisation des natures
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

data_filtered['Nature_norm'] = data_filtered['Nature'].apply(normalize_nature)

# 🎯 Données pour histogramme
cluster_labels = kmeans_TSNE_labels
df_clusters = data_filtered.copy()
df_clusters["Cluster"] = cluster_labels

# Compter les natures par cluster
nature_counts = df_clusters.groupby(['Cluster', 'Nature_norm']).size().reset_index(name='Count')
pivot_table = nature_counts.pivot(index='Cluster', columns='Nature_norm', values='Count').fillna(0)

# 🎨 Couleurs cohérentes
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 📊 Plot empilé
fig, ax = plt.subplots(figsize=(12, 7))

bottom = np.zeros(len(pivot_table))
x = np.arange(len(pivot_table))

for nature in pivot_table.columns:
    counts = pivot_table[nature].values
    bars = ax.bar(x, counts, bottom=bottom, label=nature.capitalize(), color=nature_colors.get(nature, '#999999'), edgecolor='k')

    # 🧾 Ajout des annotations dans chaque bloc
    for i, (bar, count) in enumerate(zip(bars, counts)):
        if count > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + bar.get_height() / 2,
                int(count),
                ha='center', va='center',
                fontsize=9, color='white', fontweight='bold'
            )

    bottom += counts

# 🏷️ Légendes & axes
ax.set_title("Répartition des natures par cluster (avec nombre d'incendies)")
ax.set_xlabel("Cluster")
ax.set_ylabel("Nombre d'incendies")
ax.set_xticks(x)
ax.set_xticklabels(pivot_table.index)
ax.legend(title="Nature", bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

Clusters sur carte (Kmeans)

Certains clusters omniprésents dans le nord
-> nature variée mais saison en général été car zone avec moins de risques

Dans le sud, clusters assez variés mais nature et saison variée également
-> zone à risque donc + de chance d'avoir des feux

---
###### 3.5.7.2 GMM

In [ ]:
# Copie pour éviter de modifier l'original
df_full_gmm = data_filtered.copy()

# Ajout des labels de cluster
df_full_gmm['Cluster'] = gmm_PCA_labels

In [ ]:
# Filtrer les points du cluster 1
cluster_1_data_gmm = df_full_gmm[df_full_gmm['Cluster'] == 1]

# Afficher les premières lignes pour voir ce qu’il contient
print(cluster_1_data_gmm[['Nature', 'temp_mean', 'rhum_mean', 'prec24h_mean', 'wspd_mean']].head())

In [ ]:
# Moyennes des variables météo dans le cluster 1
print("📊 Moyennes météo - Cluster 1 :")
print(cluster_1_data_gmm[['temp_mean', 'rhum_mean', 'prec24h_mean', 'wspd_mean']].mean().round(2))

# Répartition des natures dans ce cluster
print("\n🔥 Répartition des types de feu - Cluster 1 :")
print(cluster_1_data_gmm['Nature'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import unidecode

# 🔄 Normalisation des natures
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

data_filtered['Nature_norm'] = data_filtered['Nature'].apply(normalize_nature)

# 🎯 Données pour histogramme
cluster_labels = gmm_PCA_labels  # ← change selon ton clustering
df_clusters = data_filtered.copy()
df_clusters["Cluster"] = cluster_labels

# Compter les natures par cluster
nature_counts = df_clusters.groupby(['Cluster', 'Nature_norm']).size().reset_index(name='Count')
pivot_table = nature_counts.pivot(index='Cluster', columns='Nature_norm', values='Count').fillna(0)

# 🎨 Couleurs cohérentes
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 📊 Plot empilé
fig, ax = plt.subplots(figsize=(12, 7))

bottom = np.zeros(len(pivot_table))
x = np.arange(len(pivot_table))

for nature in pivot_table.columns:
    counts = pivot_table[nature].values
    bars = ax.bar(x, counts, bottom=bottom, label=nature.capitalize(), color=nature_colors.get(nature, '#999999'), edgecolor='k')

    # 🧾 Ajout des annotations dans chaque bloc
    for i, (bar, count) in enumerate(zip(bars, counts)):
        if count > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + bar.get_height() / 2,
                int(count),
                ha='center', va='center',
                fontsize=9, color='white', fontweight='bold'
            )

    bottom += counts

# 🏷️ Légendes & axes
ax.set_title("Répartition des natures par cluster (avec nombre d'incendies)")
ax.set_xlabel("Cluster")
ax.set_ylabel("Nombre d'incendies")
ax.set_xticks(x)
ax.set_xticklabels(pivot_table.index)
ax.legend(title="Nature", bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import unidecode

# 🔄 Normalisation des natures
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

data_filtered['Nature_norm'] = data_filtered['Nature'].apply(normalize_nature)

# 🎯 Données pour histogramme
cluster_labels = gmm_TSNE_labels
df_clusters = data_filtered.copy()
df_clusters["Cluster"] = cluster_labels

# Compter les natures par cluster
nature_counts = df_clusters.groupby(['Cluster', 'Nature_norm']).size().reset_index(name='Count')
pivot_table = nature_counts.pivot(index='Cluster', columns='Nature_norm', values='Count').fillna(0)

# 🎨 Couleurs cohérentes
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 📊 Plot empilé
fig, ax = plt.subplots(figsize=(12, 7))

bottom = np.zeros(len(pivot_table))
x = np.arange(len(pivot_table))

for nature in pivot_table.columns:
    counts = pivot_table[nature].values
    bars = ax.bar(x, counts, bottom=bottom, label=nature.capitalize(), color=nature_colors.get(nature, '#999999'), edgecolor='k')

    # 🧾 Ajout des annotations dans chaque bloc
    for i, (bar, count) in enumerate(zip(bars, counts)):
        if count > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + bar.get_height() / 2,
                int(count),
                ha='center', va='center',
                fontsize=9, color='white', fontweight='bold'
            )

    bottom += counts

# 🏷️ Légendes & axes
ax.set_title("Répartition des natures par cluster (avec nombre d'incendies)")
ax.set_xlabel("Cluster")
ax.set_ylabel("Nombre d'incendies")
ax.set_xticks(x)
ax.set_xticklabels(pivot_table.index)
ax.legend(title="Nature", bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

Clusters sur carte (GMM)

Certains clusters omniprésents partout (nord et sud)
-> saison et nature variée pour tous

---
###### 3.5.7.3 DBSCAN

In [ ]:
# Copie pour éviter de modifier l'original
df_full_dbscan = data_filtered.copy()

# Ajout des labels de cluster
df_full_dbscan['Cluster'] = dbscan_PCA_labels_best

In [ ]:
# Filtrer les points du cluster 1
cluster_0_data_dbscan = df_full_dbscan[df_full_dbscan['Cluster'] == 0]

# Afficher les premières lignes pour voir ce qu’il contient
print(cluster_0_data_dbscan[['Nature', 'temp_mean', 'rhum_mean', 'prec24h_mean', 'wspd_mean']].head())

In [ ]:
# Moyennes des variables météo dans le cluster 1
print("📊 Moyennes météo - Cluster 1 :")
print(cluster_0_data_dbscan[['temp_mean', 'rhum_mean', 'prec24h_mean', 'wspd_mean']].mean().round(2))

# Répartition des natures dans ce cluster
print("\n🔥 Répartition des types de feu - Cluster 1 :")
print(cluster_0_data_dbscan['Nature'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import unidecode

# 🔄 Normalisation des natures
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

data_filtered['Nature_norm'] = data_filtered['Nature'].apply(normalize_nature)

# 🎯 Données pour histogramme
cluster_labels = dbscan_PCA_labels_best  # ← change selon ton clustering
df_clusters = data_filtered.copy()
df_clusters["Cluster"] = cluster_labels

# Compter les natures par cluster
nature_counts = df_clusters.groupby(['Cluster', 'Nature_norm']).size().reset_index(name='Count')
pivot_table = nature_counts.pivot(index='Cluster', columns='Nature_norm', values='Count').fillna(0)

# 🎨 Couleurs cohérentes
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 📊 Plot empilé
fig, ax = plt.subplots(figsize=(12, 7))

bottom = np.zeros(len(pivot_table))
x = np.arange(len(pivot_table))

for nature in pivot_table.columns:
    counts = pivot_table[nature].values
    bars = ax.bar(x, counts, bottom=bottom, label=nature.capitalize(), color=nature_colors.get(nature, '#999999'), edgecolor='k')

    # 🧾 Ajout des annotations dans chaque bloc
    for i, (bar, count) in enumerate(zip(bars, counts)):
        if count > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + bar.get_height() / 2,
                int(count),
                ha='center', va='center',
                fontsize=9, color='white', fontweight='bold'
            )

    bottom += counts

# 🏷️ Légendes & axes
ax.set_title("Répartition des natures par cluster (avec nombre d'incendies)")
ax.set_xlabel("Cluster")
ax.set_ylabel("Nombre d'incendies")
ax.set_xticks(x)
ax.set_xticklabels(pivot_table.index)
ax.legend(title="Nature", bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

---
###### 3.5.7.4 Spectral

In [ ]:
# Copie pour éviter de modifier l'original
df_full_spectral = data_filtered.copy()

# Ajout des labels de cluster
df_full_spectral['Cluster'] = spectral_PCA_labels

In [ ]:
# Filtrer les points du cluster 1
cluster_0_data_spectral = df_full_spectral[df_full_spectral['Cluster'] == 0]

# Afficher les premières lignes pour voir ce qu’il contient
print(cluster_0_data_spectral[['Nature', 'temp_mean', 'rhum_mean', 'prec24h_mean', 'wspd_mean']].head())

In [ ]:
# Moyennes des variables météo dans le cluster 1
print("📊 Moyennes météo - Cluster 1 :")
print(cluster_0_data_spectral[['temp_mean', 'rhum_mean', 'prec24h_mean', 'wspd_mean']].mean().round(2))

# Répartition des natures dans ce cluster
print("\n🔥 Répartition des types de feu - Cluster 1 :")
print(cluster_0_data_spectral['Nature'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import unidecode

# 🔄 Normalisation des natures
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

data_filtered['Nature_norm'] = data_filtered['Nature'].apply(normalize_nature)

# 🎯 Données pour histogramme
cluster_labels = spectral_PCA_labels  # ← change selon ton clustering
df_clusters = data_filtered.copy()
df_clusters["Cluster"] = cluster_labels

# Compter les natures par cluster
nature_counts = df_clusters.groupby(['Cluster', 'Nature_norm']).size().reset_index(name='Count')
pivot_table = nature_counts.pivot(index='Cluster', columns='Nature_norm', values='Count').fillna(0)

# 🎨 Couleurs cohérentes
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 📊 Plot empilé
fig, ax = plt.subplots(figsize=(12, 7))

bottom = np.zeros(len(pivot_table))
x = np.arange(len(pivot_table))

for nature in pivot_table.columns:
    counts = pivot_table[nature].values
    bars = ax.bar(x, counts, bottom=bottom, label=nature.capitalize(), color=nature_colors.get(nature, '#999999'), edgecolor='k')

    # 🧾 Ajout des annotations dans chaque bloc
    for i, (bar, count) in enumerate(zip(bars, counts)):
        if count > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + bar.get_height() / 2,
                int(count),
                ha='center', va='center',
                fontsize=9, color='white', fontweight='bold'
            )

    bottom += counts

# 🏷️ Légendes & axes
ax.set_title("Répartition des natures par cluster (avec nombre d'incendies)")
ax.set_xlabel("Cluster")
ax.set_ylabel("Nombre d'incendies")
ax.set_xticks(x)
ax.set_xticklabels(pivot_table.index)
ax.legend(title="Nature", bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

Clusters sur carte (Spectral)

Certains clusters omniprésents partout
-> saison et nature variée

D'autres omniprésents dans le nord
-> saison la plus représenté = été, nature variée
-> zone avec moins de risques hors été
    -> majorité des feux dans le nord durant cette saison

---
###### 3.5.7.5 Agglomerative

In [ ]:
# Copie pour éviter de modifier l'original
df_full_agglo = data_filtered.copy()

# Ajout des labels de cluster
df_full_agglo['Cluster'] = agglo_clus_PCA_labels

In [ ]:
# Filtrer les points du cluster 1
cluster_0_data_agglo = df_full_agglo[df_full_agglo['Cluster'] == 0]

# Afficher les premières lignes pour voir ce qu’il contient
print(cluster_0_data_agglo[['Nature', 'temp_mean', 'rhum_mean', 'prec24h_mean', 'wspd_mean']].head())

In [ ]:
# Moyennes des variables météo dans le cluster 1
print("📊 Moyennes météo - Cluster 1 :")
print(cluster_0_data_agglo[['temp_mean', 'rhum_mean', 'prec24h_mean', 'wspd_mean']].mean().round(2))

# Répartition des natures dans ce cluster
print("\n🔥 Répartition des types de feu - Cluster 1 :")
print(cluster_0_data_agglo['Nature'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import unidecode

# 🔄 Normalisation des natures
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

data_filtered['Nature_norm'] = data_filtered['Nature'].apply(normalize_nature)

# 🎯 Données pour histogramme
cluster_labels = agglo_clus_PCA_labels  # ← change selon ton clustering
df_clusters = data_filtered.copy()
df_clusters["Cluster"] = cluster_labels

# Compter les natures par cluster
nature_counts = df_clusters.groupby(['Cluster', 'Nature_norm']).size().reset_index(name='Count')
pivot_table = nature_counts.pivot(index='Cluster', columns='Nature_norm', values='Count').fillna(0)

# 🎨 Couleurs cohérentes
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 📊 Plot empilé
fig, ax = plt.subplots(figsize=(12, 7))

bottom = np.zeros(len(pivot_table))
x = np.arange(len(pivot_table))

for nature in pivot_table.columns:
    counts = pivot_table[nature].values
    bars = ax.bar(x, counts, bottom=bottom, label=nature.capitalize(), color=nature_colors.get(nature, '#999999'), edgecolor='k')

    # 🧾 Ajout des annotations dans chaque bloc
    for i, (bar, count) in enumerate(zip(bars, counts)):
        if count > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + bar.get_height() / 2,
                int(count),
                ha='center', va='center',
                fontsize=9, color='white', fontweight='bold'
            )

    bottom += counts

# 🏷️ Légendes & axes
ax.set_title("Répartition des natures par cluster (avec nombre d'incendies)")
ax.set_xlabel("Cluster")
ax.set_ylabel("Nombre d'incendies")
ax.set_xticks(x)
ax.set_xticklabels(pivot_table.index)
ax.legend(title="Nature", bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import unidecode

# 🔄 Normalisation des natures
def normalize_nature(nature):
    nature = str(nature).lower().strip()
    nature = unidecode.unidecode(nature)
    nature = nature.replace('( particulier )', '(particulier)').replace('(', ' (').replace(')', ')')
    nature = nature.replace('  ', ' ').strip()
    return nature

data_filtered['Nature_norm'] = data_filtered['Nature'].apply(normalize_nature)

# 🎯 Données pour histogramme
cluster_labels = agglo_clus_TSNE_labels  # ← change selon ton clustering
df_clusters = data_filtered.copy()
df_clusters["Cluster"] = cluster_labels

# Compter les natures par cluster
nature_counts = df_clusters.groupby(['Cluster', 'Nature_norm']).size().reset_index(name='Count')
pivot_table = nature_counts.pivot(index='Cluster', columns='Nature_norm', values='Count').fillna(0)

# 🎨 Couleurs cohérentes
nature_colors = {
    'accidentelle': '#1f77b4',
    'naturelle': '#2ca02c',
    'malveillance': '#ff7f0e',
    'involontaire (particulier)': '#d62728',
    'involontaire (travaux)': '#9467bd'
}

# 📊 Plot empilé
fig, ax = plt.subplots(figsize=(12, 7))

bottom = np.zeros(len(pivot_table))
x = np.arange(len(pivot_table))

for nature in pivot_table.columns:
    counts = pivot_table[nature].values
    bars = ax.bar(x, counts, bottom=bottom, label=nature.capitalize(), color=nature_colors.get(nature, '#999999'), edgecolor='k')

    # 🧾 Ajout des annotations dans chaque bloc
    for i, (bar, count) in enumerate(zip(bars, counts)):
        if count > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + bar.get_height() / 2,
                int(count),
                ha='center', va='center',
                fontsize=9, color='white', fontweight='bold'
            )

    bottom += counts

# 🏷️ Légendes & axes
ax.set_title("Répartition des natures par cluster (avec nombre d'incendies)")
ax.set_xlabel("Cluster")
ax.set_ylabel("Nombre d'incendies")
ax.set_xticks(x)
ax.set_xticklabels(pivot_table.index)
ax.legend(title="Nature", bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

Clusters sur carte (Agglomerative)

Certains clusters omniprésents partout
-> saison et nature variée

D'autres omniprésents dans le nord
-> saison la plus représenté = été, nature variée
-> zone avec moins de risques hors été
    -> majorité des feux dans le nord durant cette saison

D'autres omniprésents dans le sud
-> zone avec le plus de risques
-> nature et saison variée

---
##### 3.5.8 Prédictions

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

scaler = StandardScaler()
X_scaled_pred = scaler.fit_transform(X)
# 📊 PCA sur les données normalisées
pca = PCA()
X_pca_full = pca.fit_transform(X_scaled_pred)

# 📈 Variance expliquée (individuelle et cumulée)
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance) * 100

# 📍 Trouver combien de composantes sont nécessaires pour 90% et 95%
nb_99 = np.argmax(cumulative_variance >= 99) + 1
nb_90 = np.argmax(cumulative_variance >= 90) + 1
nb_80 = np.argmax(cumulative_variance >= 80) + 1
nb_85 = np.argmax(cumulative_variance >= 85) + 1
nb_95 = np.argmax(cumulative_variance >= 95) + 1

# 📊 Plot de la variance expliquée cumulée
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--', color='b')
plt.axhline(y=99, color='y', linestyle='--', label='99% de variance')
plt.axhline(y=95, color='r', linestyle='--', label='95% de variance')
plt.axhline(y=90, color='y', linestyle='--', label='90% de variance')
plt.axhline(y=85, color='cyan', linestyle='--', label='85% de variance')
plt.axhline(y=80, color='g', linestyle='--', label='80% de variance')

# ✍️ Annotations
plt.annotate(f"{nb_99} composantes", (nb_99, cumulative_variance[nb_99-1]),
             textcoords="offset points", xytext=(0,10), ha='center', color='yellow')
plt.annotate(f"{nb_95} composantes", (nb_95, cumulative_variance[nb_95-1]),
             textcoords="offset points", xytext=(0,10), ha='center', color='red')
plt.annotate(f"{nb_90} composantes", (nb_90, cumulative_variance[nb_90-1]),
             textcoords="offset points", xytext=(0,10), ha='center', color='yellow')
plt.annotate(f"{nb_85} composantes", (nb_85, cumulative_variance[nb_85-1]),
             textcoords="offset points", xytext=(0,10), ha='center', color='cyan')
plt.annotate(f"{nb_80} composantes", (nb_80, cumulative_variance[nb_80-1]),
             textcoords="offset points", xytext=(0,10), ha='center', color='green')

# 🎯 Finitions
plt.title("Variance expliquée cumulée par PCA")
plt.xlabel("Nombre de composantes principales")
plt.ylabel("Variance expliquée cumulée (%)")
plt.grid(True)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# 📋 Affichage d'un tableau résumé
summary_df = pd.DataFrame({
    'Composante': range(1, len(explained_variance) + 1),
    'Variance expliquée (%)': explained_variance * 100,
    'Variance expliquée cumulée (%)': cumulative_variance
})

# Affiche les 20 premières lignes, ou plus selon tes besoins
print(summary_df.head(20))

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler, BorderlineSMOTE
import pandas as pd
import numpy as np

# 🔁 Choix du sampler
sampling_method = 'RandomOverSampler'  # 'SMOTE', 'ADASYN', 'RandomOverSampler', 'BorderlineSMOTE'

def get_sampler(method):
    if method == 'SMOTE':
        return SMOTE(random_state=42)
    elif method == 'ADASYN':
        return ADASYN(random_state=42)
    elif method == 'RandomOverSampler':
        return RandomOverSampler(random_state=42)
    elif method == 'BorderlineSMOTE':
        return BorderlineSMOTE(random_state=42)
    else:
        raise ValueError("Méthode de suréchantillonnage non reconnue.")

# 🧪 Standardisation
scaler = RobustScaler()
X_scaled = scaler.fit_transform(data_filtered[features_name].fillna(0))

# 🧬 PCA : on garde les composantes expliquant 90% de la variance
pca = PCA(n_components=81, svd_solver='full')
X_pca = pca.fit_transform(X_scaled)
print(f"✅ Nombre de composantes conservées : {pca.n_components_}")

# 🔢 Cibles binaires
data_filtered['y_acc'] = (data_filtered['Nature'] == 'Accidentelle').astype(int)
data_filtered['y_mal'] = (data_filtered['Nature'] == 'Malveillance').astype(int)
data_filtered['y_inv_par'] = (data_filtered['Nature'] == 'Involontaire (particulier)').astype(int)
data_filtered['y_inv_trav'] = (data_filtered['Nature'] == 'Involontaire (travaux)').astype(int)
data_filtered['y_nat'] = (data_filtered['Nature'] == 'Naturelle').astype(int)

y_dict = {
    'Accidentelle': data_filtered['y_acc'],
    'Malveillance': data_filtered['y_mal'],
    'Involontaire (particulier)': data_filtered['y_inv_par'],
    'Involontaire (travaux)': data_filtered['y_inv_trav'],
    'Naturelle': data_filtered['y_nat']
}

target_names = {
    'Accidentelle': 'y_acc',
    'Malveillance': 'y_mal',
    'Involontaire (particulier)': 'y_inv_par',
    'Involontaire (travaux)': 'y_inv_trav',
    'Naturelle': 'y_nat'
}

models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=4,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=200,
        learning_rate=0.01,
        max_depth=8,
        num_leaves=20,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42
    ),

    "CatBoost": CatBoostClassifier(
        iterations=350,
        learning_rate=0.01,
        depth=6,
        l2_leaf_reg=10,
        subsample=0.8,
        verbose=0,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=1,
        reg_alpha=1,
        reg_lambda=1,
        scale_pos_weight=5,  # à adapter selon le déséquilibre de la classe cible
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
}

# ✂️ Séparation train/test
data_filtered['date'] = pd.to_datetime(data_filtered['date'], errors='coerce')
train_mask = data_filtered['date'] <= '2022-12-31'
test_mask = data_filtered['date'] >= '2023-01-01'

X_train_pca = X_pca[train_mask]
X_test_pca = X_pca[test_mask]

# 📦 Initialisation
y_pred = {model_name: {} for model_name in models}
metrics = {model_name: {} for model_name in models}

# 🔁 Pour chaque nature
for label in y_dict:
    print(f"\n📌 Entraînement sur {label} avec PCA + {sampling_method}")

    y_train = data_filtered.loc[train_mask, target_names[label]]
    y_test = data_filtered.loc[test_mask, target_names[label]]

    # ✅ Suréchantillonnage
    if len(np.unique(y_train)) > 1:
        sampler = get_sampler(sampling_method)
        X_train_res, y_train_res = sampler.fit_resample(X_train_pca, y_train)
    else:
        X_train_res, y_train_res = X_train_pca, y_train

    # 🔁 Entraînement
    for model_name, model in models.items():
        model.fit(X_train_res, y_train_res)

        if len(model.classes_) > 1:
            y_pred_classes = model.predict(X_test_pca)
        else:
            y_pred_classes = np.zeros(X_test_pca.shape[0])

        # 🧮 Métriques
        metrics[model_name][label] = {
            "Accuracy": accuracy_score(y_test, y_pred_classes),
            "Precision": precision_score(y_test, y_pred_classes, zero_division=0),
            "Recall": recall_score(y_test, y_pred_classes, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred_classes, zero_division=0)
        }

# 📊 Résultats
results_list = []
for model_name, model_metrics in metrics.items():
    for label, metric_values in model_metrics.items():
        results_list.append({
            "Modèle": model_name,
            "Nature": label,
            "Accuracy": metric_values["Accuracy"],
            "Precision": metric_values["Precision"],
            "Recall": metric_values["Recall"],
            "F1 Score": metric_values["F1 Score"],
            "Sampling": sampling_method
        })

df_results_pca = pd.DataFrame(results_list)

# 💾 Export
df_results_pca.to_csv(f"../results_img_csv/resultats_PCA.csv", index=False)
print(f"\n✅ Résultats enregistrés dans 'resultats_PCA_{sampling_method}.csv'")

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
import pandas as pd
import numpy as np

# 🔁 Paramètres
undersample_fraction = 0.5  # 💡 Ajustable
smote_k_neighbors = 4       # 💡 Paramètre SMOTE (default = 5)
n_pca_components = 104

# 🧪 Standardisation
scaler = RobustScaler()
X_scaled = scaler.fit_transform(data_filtered[features_name].fillna(0))

# 🧬 PCA
pca = PCA(n_components=104, svd_solver='full')
X_pca = pca.fit_transform(X_scaled)
print(f"✅ PCA : {pca.n_components_} composantes conservées")

# 🎯 Cibles binaires
data_filtered['y_acc'] = (data_filtered['Nature'] == 'Accidentelle').astype(int)
data_filtered['y_mal'] = (data_filtered['Nature'] == 'Malveillance').astype(int)
data_filtered['y_inv_par'] = (data_filtered['Nature'] == 'Involontaire (particulier)').astype(int)
data_filtered['y_inv_trav'] = (data_filtered['Nature'] == 'Involontaire (travaux)').astype(int)
data_filtered['y_nat'] = (data_filtered['Nature'] == 'Naturelle').astype(int)

y_dict = {
    'Accidentelle': data_filtered['y_acc'],
    'Malveillance': data_filtered['y_mal'],
    'Involontaire (particulier)': data_filtered['y_inv_par'],
    'Involontaire (travaux)': data_filtered['y_inv_trav'],
    'Naturelle': data_filtered['y_nat']
}

target_names = {
    'Accidentelle': 'y_acc',
    'Malveillance': 'y_mal',
    'Involontaire (particulier)': 'y_inv_par',
    'Involontaire (travaux)': 'y_inv_trav',
    'Naturelle': 'y_nat'
}

models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=4,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=200,
        learning_rate=0.01,
        max_depth=8,
        num_leaves=20,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42
    ),

    "CatBoost": CatBoostClassifier(
        iterations=350,
        learning_rate=0.01,
        depth=6,
        l2_leaf_reg=10,
        subsample=0.8,
        verbose=0,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=1,
        reg_alpha=1,
        reg_lambda=1,
        scale_pos_weight=5,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
}

# 📅 Split train/test
data_filtered['date'] = pd.to_datetime(data_filtered['date'], errors='coerce')
train_mask = data_filtered['date'] <= '2022-12-31'
test_mask = data_filtered['date'] >= '2023-01-01'

X_train_pca = X_pca[train_mask]
X_test_pca = X_pca[test_mask]

# 📦 Résultats
metrics = {model_name: {} for model_name in models}

def under_and_smote(X, y, fraction=0.5, k_neighbors=5):
    df = pd.DataFrame(X)
    df['target'] = y.values if isinstance(y, pd.Series) else y

    class_counts = df['target'].value_counts()
    if len(class_counts) < 2:
        return X, y  # rien à faire

    maj = class_counts.idxmax()
    min = class_counts.idxmin()

    df_maj = df[df['target'] == maj]
    df_min = df[df['target'] == min]

    n_samples = int(len(df_maj) * fraction)
    df_maj_under = df_maj.sample(n_samples, random_state=42)

    df_combined = pd.concat([df_maj_under, df_min], axis=0).sample(frac=1, random_state=42)

    X_temp = df_combined.drop(columns='target').values
    y_temp = df_combined['target'].values

    smote = SMOTE(k_neighbors=k_neighbors, random_state=42)
    X_final, y_final = smote.fit_resample(X_temp, y_temp)

    return X_final, y_final

# 🔁 Pour chaque classe cible
for label in y_dict:
    print(f"\n📌 Under + SMOTE sur '{label}' — under_fraction={undersample_fraction}")

    y_train = data_filtered.loc[train_mask, target_names[label]]
    y_test = data_filtered.loc[test_mask, target_names[label]]

    if len(np.unique(y_train)) > 1:
        X_train_final, y_train_final = under_and_smote(X_train_pca, y_train, undersample_fraction, smote_k_neighbors)
    else:
        X_train_final, y_train_final = X_train_pca, y_train

    # 🔁 Entraînement
    for model_name, model in models.items():
        model.fit(X_train_final, y_train_final)
        y_pred = model.predict(X_test_pca)

        metrics[model_name][label] = {
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0)
        }

# 📊 Résultats
results_list = []
for model_name, model_metrics in metrics.items():
    for label, metric_values in model_metrics.items():
        results_list.append({
            "Modèle": model_name,
            "Nature": label,
            "Accuracy": metric_values["Accuracy"],
            "Precision": metric_values["Precision"],
            "Recall": metric_values["Recall"],
            "F1 Score": metric_values["F1 Score"],
            "UnderSample_Frac": undersample_fraction,
            "SMOTE_k": smote_k_neighbors
        })

df_results_mix = pd.DataFrame(results_list)
df_results_mix.to_csv("../results_img_csv/resultats_PCA_UnderSampling.csv", index=False)
print("\n✅ Résultats sauvegardés dans 'resultats_PCA_UnderOver.csv'")

In [ ]:
df_results_mix

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Chargement des résultats
df_results_pca = pd.read_csv("../results_img_csv/resultats_PCA_UnderSampling.csv")

# 📦 Liste des métriques à afficher
metrics_list = ["Accuracy", "Precision", "Recall", "F1 Score"]

# 🎨 Couleurs personnalisées
colors = {
    "Accuracy": "#1f77b4",
    "Precision": "#ff7f0e",
    "Recall": "#2ca02c",
    "F1 Score": "#d62728"
}

# 🔁 Boucle sur chaque modèle
for model in df_results_pca["Modèle"].unique():
    df_model = df_results_pca[df_results_pca["Modèle"] == model]

    # Création du graphique
    fig, ax = plt.subplots(figsize=(10, 6))

    bar_width = 0.2
    x = np.arange(len(df_model["Nature"]))

    for i, metric in enumerate(metrics_list):
        ax.bar(
            x + i * bar_width,
            df_model[metric],
            width=bar_width,
            label=metric,
            color=colors[metric]
        )

    ax.set_title(f"📊 Performances du modèle : {model}", fontsize=14, fontweight='bold')
    ax.set_xlabel("Type de feu (Nature)", fontsize=12)
    ax.set_ylabel("Score", fontsize=12)
    ax.set_xticks(x + bar_width * (len(metrics_list) - 1) / 2)
    ax.set_xticklabels(df_model["Nature"], rotation=45)
    ax.set_ylim(0, 1)
    ax.legend(title="Métrique")

    plt.tight_layout()
    plt.show()

---
#### 3.6 Visu graphique autres résultats

---
##### 3.6.1 Moyenne globale (tous les deps)

In [ ]:
data_filtered['date'] = pd.to_datetime(data_filtered['date'])

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 📌 Convertir la colonne 'Date' en années (si ce n'est pas déjà le cas)
data_filtered['Year'] = data_filtered['date'].dt.year

# 📌 Calculer le nombre de feux par année et par nature
fire_counts = data_filtered.groupby(['Year', 'Nature']).agg({'nombre_feux': 'sum'}).reset_index()

# 📌 Créer un graphique
plt.figure(figsize=(12, 6))

# 📌 Ajouter une ligne pour chaque nature de feu
for nature in fire_counts['Nature'].unique():
    data_nature = fire_counts[fire_counts['Nature'] == nature]
    plt.plot(data_nature['Year'], data_nature['nombre_feux'], label=nature, marker='o')

# 📌 Ajouter des labels et un titre
plt.title("Nombre de feux par année et par nature", fontsize=16)
plt.xlabel("Année", fontsize=12)
plt.ylabel("Nombre de feux", fontsize=12)

# 📌 Ajouter une légende
plt.legend(title='Nature des feux')

# 📌 Afficher le graphique
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

data_filtered['date'] = pd.to_datetime(data_filtered['date'])
data_filtered['Year'] = data_filtered['date'].dt.year
mean_probs_by_year = data_filtered.groupby('Year')[['Accidentelle', 'Malveillance', 'Involontaire (particulier)', 'Involontaire (travaux)', 'Naturelle']].mean()

# 📊 Tracer le graphique
plt.figure(figsize=(10, 6))

# Tracer chaque nature
for nature in mean_probs_by_year.columns:
    plt.plot(mean_probs_by_year.index, mean_probs_by_year[nature], label=nature, marker='o')

# Personnaliser le graphique
plt.title("Moyenne des probabilités par année pour chaque nature d'incendie")
plt.xlabel("Année")
plt.ylabel("Probabilité")
plt.legend(title="Nature d'incendie")
plt.grid(True)

# Afficher le graphique
plt.tight_layout()
plt.show()

In [ ]:
# Ajout d'une colonne Saison si pas encore faite
def get_saison(mois):
    if mois in [12, 1, 2]:
        return 'Hiver'
    elif mois in [3, 4, 5]:
        return 'Printemps'
    elif mois in [6, 7, 8]:
        return 'Été'
    else:
        return 'Automne'

data_filtered['Mois'] = pd.to_datetime(data_filtered['date']).dt.month
data_filtered['Saison'] = data_filtered['Mois'].apply(get_saison)

# Groupement
saison_nature = data_filtered.groupby(['Saison', 'Nature']).size().reset_index(name='Nombre de feux')
saison_pivot = saison_nature.pivot(index='Saison', columns='Nature', values='Nombre de feux').fillna(0)

# Affichage
saison_pivot.plot(kind='bar', stacked=True, figsize=(10, 6))
plt.title("🔥 Répartition des types de feux par saison")
plt.ylabel("Nombre de feux")
plt.xlabel("Saison")
plt.legend(title="Nature du feu")
plt.tight_layout()
plt.grid(axis='y')
plt.show()

---
##### 3.6.2 Moyennes temporel data météorologique

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 🕒 S'assurer que 'date' est bien en datetime
data_filtered['date'] = pd.to_datetime(data_filtered['date'], errors='coerce')

# 📅 Extraire année et mois, puis grouper
data_filtered['year_month'] = data_filtered['date'].dt.to_period('M')
monthly_temp_mean = data_filtered.groupby('year_month')['temp_mean'].mean()

# 🔧 Conversion des index en datetime pour faciliter le formatage
monthly_temp_mean.index = monthly_temp_mean.index.to_timestamp()

# 📊 Bar chart avec labels simplifiés
plt.figure(figsize=(18, 6))
plt.bar(monthly_temp_mean.index, monthly_temp_mean.values, color='cornflowerblue', width=20)

plt.title("🌡️ Température moyenne par mois", fontsize=14)
plt.xlabel("Année", fontsize=12)
plt.ylabel("Température moyenne", fontsize=12)

# 🗓️ Afficher seulement l'année une fois par an
years = monthly_temp_mean.index.to_series().dt.year
xticks_positions = monthly_temp_mean.index[years.duplicated(keep='first') == False]
xtick_labels = [str(year) for year in xticks_positions.year]

plt.xticks(xticks_positions, xtick_labels, rotation=0, fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 🕒 S'assurer que 'date' est bien en datetime
data_filtered['date'] = pd.to_datetime(data_filtered['date'], errors='coerce')

# 📅 Extraire année et mois, puis grouper
data_filtered['year_month'] = data_filtered['date'].dt.to_period('M')
monthly_rhum_mean = data_filtered.groupby('year_month')['rhum_mean'].mean()

# 🔧 Conversion des index en datetime pour faciliter le formatage
monthly_rhum_mean.index = monthly_rhum_mean.index.to_timestamp()

# 📊 Bar chart avec labels simplifiés
plt.figure(figsize=(18, 6))
plt.bar(monthly_rhum_mean.index, monthly_rhum_mean.values, color='cornflowerblue', width=20)

plt.title("🌡️ Humidité moyenne par mois", fontsize=14)
plt.xlabel("Année", fontsize=12)
plt.ylabel("Humidité moyenne", fontsize=12)

# 🗓️ Afficher seulement l'année une fois par an
years = monthly_rhum_mean.index.to_series().dt.year
xticks_positions = monthly_rhum_mean.index[years.duplicated(keep='first') == False]
xtick_labels = [str(year) for year in xticks_positions.year]

plt.xticks(xticks_positions, xtick_labels, rotation=0, fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()